# Radiant RAG MCP — Video Ingestion & Summarization Test Notebook

End-to-end test of the `ingest_video` MCP tool and `VideoSummarizationAgent`.
Synthetic test videos are generated in-process so the suite runs without any
external media files.

| | |
|---|---|
| **Transport** | Streamable HTTP |
| **Backend** | ChromaDB |
| **LLM** | Ollama Cloud — set `OLLAMA_API_KEY` in Colab Secrets |
| **VLM** | `Qwen/Qwen2-VL-2B-Instruct` (T4 GPU recommended) |

| # | Section | LLM | VLM |
|---|---------|-----|-----|
| 1 | Install video extras | | |
| 2 | Import checks | | |
| 3 | Configuration display | | |
| 4 | Shared helpers (client, pretty-print) | | |
| 5 | Server startup | | |
| 6 | Tool list verification | | |
| 7 | Create synthetic test videos | | |
| 8 | `_has_audio()` smoke test | | |
| 9 | `ingest_video` — audio path | | |
| 10 | `ingest_video` — silent path | | VLM |
| 11 | `search_knowledge` across video content | | |
| 12 | `query_knowledge` grounded in video | LLM | |
| 13 | `VideoSummarizationAgent` — `brief` | LLM | |
| 14 | `VideoSummarizationAgent` — `standard` | LLM | |
| 15 | `VideoSummarizationAgent` — `detailed` | LLM | |
| 16 | Word-count comparison (window / chapter / overall) | LLM | |
| 17 | YouTube URL ingestion | LLM | |
| 18 | Cleanup | | |

## 1  Install

Installs the core `radiant-rag-mcp` package and all video dependencies,
then resolves the two distribution gaps that cause `ingest_video` to be absent.

**The video source files do not exist on `main`.**
Set `VIDEO_BRANCH` below to the branch where `video_processor.py` and
`video_summarization.py` live (e.g. `"feature/video"`, `"dev"`, `"video-support"`).
The cell probes a list of candidate branches automatically, so if you leave
`VIDEO_BRANCH = ""` it will try them all and use the first one that has the files.


In [ ]:
import subprocess, sys, os

CONFIG_PATH = '/content/config.yaml'

# ── Step 0: Install FFmpeg 6 via PPA (BEFORE pip installs) ─────────────────────
# torchcodec needs libavutil.so.58 (FFmpeg 6). The default Ubuntu apt only has
# FFmpeg 4 (libavutil.so.56). Install FFmpeg 6 from the ubuntuhandbook1 PPA first
# so the shared libraries exist when torchcodec is loaded after pip installs.
!add-apt-repository -y ppa:ubuntuhandbook1/ffmpeg6 > /dev/null 2>&1
!apt-get update -qq
!apt-get install -y -q ffmpeg
!ffmpeg -version 2>&1 | head -1

# ── Step 1: Detect PyTorch CUDA tag before any reinstalls ────────────────────
import torch as _torch
_torch_cuda = _torch.version.cuda or ''
_cu_tag = f"cu{_torch_cuda.replace('.','')[:3]}" if _torch_cuda else ''
print(f'torch {_torch.__version__}  CUDA {_torch_cuda}  ->  tag: {_cu_tag}')
del _torch

# ── Step 2: Core RAG package ──────────────────────────────────────────────────
# [chroma] already bundles: torch, transformers, sentence-transformers,
# qwen-vl-utils, yt-dlp, faster-whisper, accelerate, Pillow, and more.
!pip install -q --upgrade --no-cache-dir "radiant-rag-mcp[chroma] @ git+https://github.com/dshipley71/radiant-rag-mcp.git"
!pip install -q --prefer-binary nest_asyncio httpx "fastmcp>=3.0"
!wget -q "https://raw.githubusercontent.com/dshipley71/radiant-rag-mcp/main/config.yaml" -O {CONFIG_PATH}

# ── Step 3: Video extras not bundled in [chroma] ─────────────────────────────
!pip install -q opencv-python-headless ffmpeg-python "moviepy>=1.0.3" "scenedetect[opencv]>=0.6.3"
!pip install -q fasttext-wheel 2>/dev/null || true

# ── Step 4: Repair numpy/scipy/sklearn + upgrade transformers ─────────────────
# --force-reinstall makes numpy self-consistent (prevents _center ABI errors).
# transformers>=4.45.0 is required for Qwen2VLForConditionalGeneration (VLM).
!pip install -q --force-reinstall numpy scipy scikit-learn
!pip install -q --upgrade "transformers>=4.45.0" "sentence-transformers>=2.7"

# ── Step 4b: Reinstall torchcodec matching PyTorch CUDA ──────────────────────────
# sentence-transformers pulls torchcodec which needs PyTorch ABI + FFmpeg 6.
# Reinstall from the PyTorch whl index to fix the MessageLogger ABI mismatch.
import subprocess as _sp2, sys
if _cu_tag:
    _r = _sp2.run(
        [sys.executable, '-m', 'pip', 'install', '-q',
         '--force-reinstall', 'torchcodec',
         '--index-url', f'https://download.pytorch.org/whl/{_cu_tag}'],
        capture_output=True, text=True
    )
    if _r.returncode == 0:
        print(f'torchcodec ({_cu_tag})  OK')
    else:
        print(f'torchcodec install failed: {_r.stderr[:200]}')
        print('  torchcodec will be skipped by sentence-transformers at runtime.')
del _sp2

# ── Step 5: torchvision matching the installed CUDA tag ───────────────────────
if _cu_tag:
    _whl = f'https://download.pytorch.org/whl/{_cu_tag}'
    print(f'Installing torchvision from {_whl} ...')
    _r = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q',
         '--force-reinstall', 'torchvision', '--index-url', _whl],
        capture_output=True, text=True
    )
    if _r.returncode == 0:
        print(f'torchvision ({_cu_tag})  OK')
    else:
        print(f'  {_cu_tag} wheel not found, falling back to PyPI ...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                        '--force-reinstall', 'torchvision'], check=True)
        print('torchvision (PyPI fallback)  OK')

# ── Step 6: Pillow >=11.0 after torchvision (PIL._typing._Ink added in 11.0) ──
!pip install -q --force-reinstall "Pillow>=11.0"

# ── Step 7: Restart so freshly installed .so files are loaded ─────────────────
print("All installs complete. Restarting kernel ...")
import time; time.sleep(1)
os.kill(os.getpid(), 9)


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists...
Building dependency tree...
Reading state information...
The following additional packages will be installed:
  libavcodec60 libavdevice60 libavfilter9 libavformat60 libavutil58 libcjson1
  libdav1d6 libhwy1 libjxl0.7 liblcms2-2 libpostproc57 librist4 libsvtav1enc1
  libswresample4 libswscale7 libvidstab1.2 libvpl2
Suggested packages:
  ffmpeg-doc libcuda1 libnvcuvid1 libnvidia-encode1 liblcms2-utils
The following NEW packages will be installed:
  libavcodec60 libavdevice60 libavfilter9 libavformat60 libavutil58 libcjson1
  libdav1d6 libhwy1 libjxl0.7 libpostproc57 librist4 libsvtav1enc1
  libswresample4 libswscale7 libvidstab1.2 libvpl2
The following packages will be upgraded:
  ffmpeg liblcms2-2
2 upgraded, 16 newly installed, 0 to remove and 107 not upgraded.
Need to get 1

### ↻ Kernel restarted — continue from cell 1b

All package repairs completed. Run **cell 1b** to verify imports, cache models, and fetch video files.


In [1]:
# ── Cell 1b: runs after the automatic kernel restart ─────────────────────────
import sys, importlib.util, warnings
from pathlib import Path
warnings.filterwarnings('ignore', category=SyntaxWarning, module='moviepy')

CONFIG_PATH = '/content/config.yaml'

# 1 ── Verify numpy ────────────────────────────────────────────────────────────
import numpy as np
print(f'numpy  {np.__version__}')
try:
    import numpy._core.strings
    print('numpy  OK')
except (ImportError, AttributeError) as _e:
    raise RuntimeError(f'numpy inconsistent ({_e}). Re-run 1a → restart → re-run 1b.')

# 2 ── Verify scipy ────────────────────────────────────────────────────────────
import scipy
print(f'scipy  {scipy.__version__}  OK')

# 3 ── Verify torchvision + Pillow ─────────────────────────────────────────────
import torchvision, PIL
print(f'torchvision  {torchvision.__version__}  OK')
print(f'Pillow       {PIL.__version__}  OK')

# 4 ── Pre-cache embedding / reranking models ──────────────────────────────────
from sentence_transformers import SentenceTransformer, CrossEncoder
SentenceTransformer('sentence-transformers/all-MiniLM-L12-v2')
print('sentence-transformers  OK')
CrossEncoder('cross-encoder/ms-marco-MiniLM-L12-v2')
print('cross-encoder  OK')

# 5 ── Add cloud vision support via subclass (no file patching) ────────────────
# Instead of modifying the installed image_captioner.py (which risks corrupting
# it), we subclass OllamaVLMCaptioner and monkey-patch create_captioner.
# CloudVLMCaptioner overrides is_available() and caption_image() to use the
# OpenAI-compatible /chat/completions endpoint for HTTPS Ollama cloud URLs.
import os, base64
from radiant_rag_mcp.ingestion import image_captioner as _ic_mod
from radiant_rag_mcp.ingestion.image_captioner import OllamaVLMCaptioner, VLMConfig
import logging as _log

class CloudVLMCaptioner(OllamaVLMCaptioner):
    """OllamaVLMCaptioner extended for cloud OpenAI-compatible vision endpoints."""

    def __init__(self, config, ollama_url='https://ollama.com/v1', model='gemma3:12b-cloud'):
        super().__init__(config, ollama_url=ollama_url, model=model)
        self._api_key = (os.environ.get('RADIANT_OLLAMA_API_KEY', '')
                         or os.environ.get('OLLAMA_API_KEY', ''))

    def is_available(self):
        if self._available is not None:
            return self._available
        if not self._config.enabled:
            self._available = False
            return False
        # Cloud: available when API key and model are configured
        if self._ollama_url.startswith('https://'):
            self._available = bool(self._api_key and self._model)
            if self._available:
                _log.getLogger(__name__).info(
                    f'Cloud VLM configured: {self._model} @ {self._ollama_url}')
            return self._available
        return super().is_available()

    def caption_image(self, image_path, prompt=None):
        if not self.is_available() or not os.path.exists(image_path):
            return None
        prompt = prompt or self._config.caption_prompt
        try:
            import requests
            with open(image_path, 'rb') as _f:
                _b64 = base64.b64encode(_f.read()).decode('utf-8')
            if self._ollama_url.startswith('https://'):
                _resp = requests.post(
                    f'{self._ollama_url}/chat/completions',
                    headers={'Authorization': f'Bearer {self._api_key}',
                             'Content-Type': 'application/json'},
                    json={
                        'model': self._model,
                        'stream': False,
                        'max_tokens': self._config.max_new_tokens,
                        'messages': [{'role': 'user', 'content': [
                            {'type': 'text', 'text': prompt},
                            {'type': 'image_url',
                             'image_url': {'url': f'data:image/jpeg;base64,{_b64}'}}
                        ]}]
                    },
                    timeout=120)
                if _resp.status_code == 200:
                    return (_resp.json().get('choices', [{}])[0]
                            .get('message', {}).get('content', '').strip()) or None
                _log.getLogger(__name__).error(
                    f'Cloud VLM HTTP {_resp.status_code}: {_resp.text[:200]}')
                return None
            return super().caption_image(image_path, prompt)
        except Exception as e:
            _log.getLogger(__name__).error(f'CloudVLMCaptioner error: {e}')
            return None


def _cloud_create_captioner(vlm_config=None, ollama_url=None, ollama_model=None):
    """Replacement for create_captioner that prefers CloudVLMCaptioner for HTTPS."""
    cfg = vlm_config or VLMConfig()
    if not cfg.enabled:
        return None
    url   = (ollama_url   or os.environ.get('RADIANT_VLM_OLLAMA_FALLBACK_URL',
                                             'https://ollama.com/v1'))
    model = (ollama_model or os.environ.get('RADIANT_VLM_OLLAMA_FALLBACK_MODEL',
                                             cfg.model_name))
    if url.startswith('https://'):
        cap = CloudVLMCaptioner(cfg, ollama_url=url, model=model)
        if cap.is_available():
            return cap
    # Fall back to original behaviour
    return _orig_create_captioner(vlm_config=cfg, ollama_url=ollama_url,
                                  ollama_model=ollama_model)


# Monkey-patch create_captioner in the module and its public re-export
_orig_create_captioner = _ic_mod.create_captioner
_ic_mod.create_captioner = _cloud_create_captioner
# Also patch in app.py's already-imported reference
try:
    import radiant_rag_mcp.app as _app_mod
    _app_mod.create_captioner = _cloud_create_captioner
except Exception:
    pass

print('CloudVLMCaptioner registered — create_captioner patched for Ollama cloud.')
print('\nInstall complete')


numpy  2.4.3
numpy  OK
scipy  1.17.1  OK
torchvision  0.26.0+cu128  OK
Pillow       12.2.0  OK


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

cross-encoder  OK


/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:294: SyntaxWarning: invalid escape sequence '\d'
  lines_video = [l for l in lines if ' Video: ' in l and re.search('\d+x\d+', l)]
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:367: SyntaxWarning: invalid escape sequence '\d'
  rotation_lines = [l for l in lines if 'rotate          :' in l and re.search('\d+$', l)]
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:370: SyntaxWarning: invalid escape sequence '\d'
  match = re.search('\d+$', rotation_line)
/usr/local/lib/python3.12/dist-packages/moviepy/config_defaults.py:47: SyntaxWarning: invalid escape sequence '\P'
  IMAGEMAGICK_BINARY = r"C:\Program Files\ImageMagick-6.8.8-Q16\magick.exe"


CloudVLMCaptioner registered — create_captioner patched for Ollama cloud.

Install complete


## 2  Import checks

Verifies every dependency the video pipeline needs is importable.
A `MISSING` line means the package failed to install in section 1.

In [2]:
import sys
import importlib
import traceback
from pathlib import Path

# ── Required third-party packages (hard-fail) ─────────────────────────────────
REQUIRED = [
    ('fastmcp',               'fastmcp'),
    ('nest_asyncio',          'nest_asyncio'),
    ('chromadb',              'chromadb'),
    ('sentence_transformers', 'sentence-transformers'),
    ('yt_dlp',                'yt-dlp'),
    ('faster_whisper',        'faster-whisper'),
    ('cv2',                   'opencv-python-headless'),
    ('PIL',                   'Pillow'),
    ('scenedetect',           'scenedetect[opencv]'),
    ('ffmpeg',                'ffmpeg-python'),
    ('transformers',          'transformers'),
]

# ── Radiant video submodules (soft-fail with full traceback diagnostic) ────────
VIDEO_CORE = [
    ('radiant_rag_mcp',                            'radiant-rag-mcp'),
    ('radiant_rag_mcp.ingestion.video_processor',  'see diagnostic below'),
    ('radiant_rag_mcp.agents.video_summarization', 'see diagnostic below'),
]

_missing_required = []
_missing_video    = {}   # module -> exception

print('=== Required packages ===')
for module, pkg in REQUIRED:
    try:
        __import__(module)
        print(f'  ✓       {module}')
    except ImportError as e:
        print(f'  ✗  {module}  ->  pip install {pkg}')
        _missing_required.append(pkg)

print()
print('=== Radiant RAG video submodules ===')
for module, _ in VIDEO_CORE:
    try:
        __import__(module)
        print(f'  ✓       {module}')
    except Exception as e:
        print(f'  ✗     {module}')
        _missing_video[module] = e

# ── Diagnostic: print the real import error for each failing submodule ─────────
if _missing_video:
    print()
    print('  ─── Import error details ───────────────────────────────────────────')
    for mod, exc in _missing_video.items():
        print(f'  Module : {mod}')
        print(f'  Error  : {type(exc).__name__}: {exc}')
        # Walk to the root cause (ImportError chain)
        cause = exc.__cause__ or exc.__context__
        if cause:
            print(f'  Caused by: {type(cause).__name__}: {cause}')
        print()

    # Show which files exist on disk so we know if it's missing vs broken
    try:
        import radiant_rag_mcp as _pkg
        _root = Path(_pkg.__file__).parent
        print(f'  Package root: {_root}')
        for subpath in ['ingestion', 'ingestion/video_processor.py',
                        'agents',   'agents/video_summarization.py']:
            p = _root / subpath
            print(f'  {"exists" if p.exists() else "ABSENT "}  {p}')
    except Exception as probe_err:
        print(f'  Package probe failed: {probe_err}')

    print()
    print('  ► The most common root cause is a missing top-level import inside')
    print('    video_processor.py or video_summarization.py (e.g. moviepy,')
    print('    transformers, faster_whisper).  The error line above identifies it.')
    print('  ► Install the missing package, then restart the runtime and re-run')
    print('    sections 1 and 2.')

if _missing_required:
    raise ImportError(f'Missing required packages: {", ".join(_missing_required)}')

if _missing_video:
    import warnings
    warnings.warn(
        f'Video submodules not importable: {list(_missing_video)}.  '
        'Sections 8-18 will fail.  See diagnostic above.',
        stacklevel=1,
    )
else:
    print()
    print('All imports ok')


=== Required packages ===
  ✓       fastmcp
  ✓       nest_asyncio
  ✓       chromadb
  ✓       sentence_transformers
  ✓       yt_dlp
  ✓       faster_whisper
  ✓       cv2
  ✓       PIL
  ✓       scenedetect
  ✓       ffmpeg
  ✓       transformers

=== Radiant RAG video submodules ===
  ✓       radiant_rag_mcp
  ✓       radiant_rag_mcp.ingestion.video_processor
  ✓       radiant_rag_mcp.agents.video_summarization

All imports ok


## 3  Configuration

Sets every environment variable the server needs before it starts.
Video-specific keys use the prefix `RADIANT_VIDEO_*`.
LLM and VLM settings are displayed so they are easy to spot-check.

> Add `OLLAMA_API_KEY` to **Colab Secrets** (key icon in the left sidebar)
> before running `[LLM]` cells.

In [3]:
import os

# Read LLM key from Colab Secrets, falling back to env
try:
    from google.colab import userdata
    _key = userdata.get('OLLAMA_API_KEY') or ''
except Exception:
    _key = os.environ.get('RADIANT_OLLAMA_API_KEY', '')

# Core server settings
os.environ.update({
    'RADIANT_OLLAMA_BASE_URL':               'https://ollama.com/v1',
    'RADIANT_OLLAMA_API_KEY':                _key,
    'RADIANT_TRANSPORT':                     'http',
    'RADIANT_HOST':                          '127.0.0.1',
    'RADIANT_PORT':                          '8765',
    'RADIANT_STORAGE_BACKEND':               'chroma',
    'RADIANT_CONFIG_PATH':                   CONFIG_PATH,
    # Pipeline
    'RADIANT_PIPELINE_USE_CRITIC':           'false',
    'RADIANT_CITATION_ENABLED':              'false',
    'RADIANT_LLM_BACKEND_TIMEOUT':           '120',
    'RADIANT_LLM_BACKEND_MAX_RETRIES':       '0',
    'RADIANT_RERANKING_BACKEND_DEVICE':      'cuda',
    'RADIANT_EMBEDDING_BACKEND_DEVICE':      'cuda',
    'RADIANT_CHUNKING_USE_LLM_CHUNKING':     'false',
    'RADIANT_RETRIEVAL_DENSE_TOP_K':         '5',
    'RADIANT_RETRIEVAL_BM25_TOP_K':          '5',
    # VLM — use Ollama cloud (same service as LLM, no local GPU/model needed)
    'RADIANT_VLM_ENABLED':                   'true',
    'RADIANT_VLM_MODEL_NAME':                'gemma3:12b-cloud',
    'RADIANT_VLM_DEVICE':                    'cloud',          # sentinel for cloud Ollama (device field unused)
    'RADIANT_VLM_OLLAMA_FALLBACK_URL':       'https://ollama.com/v1',
    'RADIANT_VLM_OLLAMA_FALLBACK_MODEL':     'gemma3:12b-cloud',
    # Video processor
    'RADIANT_VIDEO_WHISPER_MODEL':                  'base',
    'RADIANT_VIDEO_WINDOW_DURATION_SECONDS':        '10.0',
    'RADIANT_VIDEO_WINDOW_OVERLAP_SECONDS':         '2.0',
    'RADIANT_VIDEO_FRAMES_PER_WINDOW':              '3',
    'RADIANT_VIDEO_ENABLE_SCENE_CHANGE_DETECTION':  'true',
    'RADIANT_VIDEO_SCENE_CHANGE_THRESHOLD':         '0.25',
    'RADIANT_VIDEO_FILMSTRIP_TILE_WIDTH':           '480',
    'RADIANT_VIDEO_FILMSTRIP_TILE_HEIGHT':          '270',
    'RADIANT_VIDEO_ENABLE_SILENT_VIDEO_ANALYSIS':   'true',
    # Summarization defaults (overridden per-cell in sections 13-16)
    'RADIANT_VIDEO_SUMMARIZATION_SUMMARY_DETAIL':           'standard',
    'RADIANT_VIDEO_SUMMARIZATION_WINDOW_CAPTION_SENTENCES': '4',
})

SERVER_URL  = f"http://{os.environ['RADIANT_HOST']}:{os.environ['RADIANT_PORT']}/mcp"
HAS_LLM_KEY = bool(_key)

print('=== Configuration ===')
print(f'  Server URL      : {SERVER_URL}')
print(f'  LLM key set     : {HAS_LLM_KEY}')
print(f'  VLM model       : {os.environ["RADIANT_VLM_MODEL_NAME"]} (via Ollama cloud)')
print(f'  Whisper model   : {os.environ["RADIANT_VIDEO_WHISPER_MODEL"]}')
print(f'  Window duration : {os.environ["RADIANT_VIDEO_WINDOW_DURATION_SECONDS"]}s')
print(f'  Frames/window   : {os.environ["RADIANT_VIDEO_FRAMES_PER_WINDOW"]}')
print(f'  Summary detail  : {os.environ["RADIANT_VIDEO_SUMMARIZATION_SUMMARY_DETAIL"]}')

if not HAS_LLM_KEY:
    print()
    print('[NOTE] OLLAMA_API_KEY not found. [LLM] cells will be skipped.')
    print('       Add it to Colab Secrets (key icon, left sidebar).')

# ── VLM availability: use create_captioner (same code path as the server) ─────
try:
    from radiant_rag_mcp.ingestion.image_captioner import (
        TRANSFORMERS_AVAILABLE as _TA, TORCH_AVAILABLE as _TorchA,
        create_captioner as _cc, VLMConfig as _VC,
    )
    _cap = _cc(vlm_config=_VC(
        enabled=True,
        model_name=os.environ['RADIANT_VLM_MODEL_NAME'],
        device=os.environ.get('RADIANT_VLM_DEVICE', 'cloud'),
    ))
    HAS_VLM = _cap is not None
    if HAS_VLM:
        print(f'VLM  OK  (TRANSFORMERS={_TA}  TORCH={_TorchA})')
    else:
        print(f'[NOTE] VLM captioner=None (TRANSFORMERS={_TA}  TORCH={_TorchA})')
        print('       Sections 10, 13-16 require VLM and will be skipped.')
    del _TA, _TorchA, _cc, _VC, _cap
except Exception as _e:
    HAS_VLM = False
    print(f'[NOTE] VLM check failed: {_e}')
    print('       Sections 10, 13-16 will be skipped.')


=== Configuration ===
  Server URL      : http://127.0.0.1:8765/mcp
  LLM key set     : True
  VLM model       : gemma3:12b-cloud (via Ollama cloud)
  Whisper model   : base
  Window duration : 10.0s
  Frames/window   : 3
  Summary detail  : standard
VLM  OK  (TRANSFORMERS=False  TORCH=True)


## 4  Shared helpers

Defines:
- `run(tool, args)` — call an MCP tool and pretty-print the JSON result
- `assert_ok(result)` — raise `AssertionError` on tool errors
- `skip_llm()` — guard used in every `[LLM]` cell
- `_wait_for_server()` — async HTTP probe used by section 5

In [4]:
import asyncio
import json
import logging
import threading
import time
import textwrap
from pathlib import Path

import nest_asyncio
nest_asyncio.apply()  # Allow asyncio.run() inside Jupyter/Colab event loop

import httpx as _httpx
from fastmcp import Client
from fastmcp.exceptions import ToolError


async def _call(tool: str, args: dict = None):
    """Async helper: open a short-lived MCP client, call the tool, return parsed result."""
    try:
        async with Client(SERVER_URL) as client:
            raw = await client.call_tool(tool, args or {})
    except ToolError as e:
        return {'status': 'error', 'tool': tool, 'message': str(e)}
    except Exception as e:
        return {'status': 'error', 'tool': tool, 'message': f'{type(e).__name__}: {e}'}

    content = raw.content if hasattr(raw, 'content') else (raw or [])
    if not content:
        return None
    item = content[0]
    text = item.text if hasattr(item, 'text') else str(item)
    try:
        return json.loads(text)
    except (json.JSONDecodeError, ValueError):
        return text


def run(tool: str, args: dict = None):
    """Synchronous wrapper around _call; pretty-prints and returns the result."""
    result = asyncio.run(_call(tool, args))
    print(json.dumps(result, indent=2, default=str))
    return result


def skip_llm() -> bool:
    """Return True (and print a notice) when no LLM key is available."""
    if not HAS_LLM_KEY:
        print('[SKIPPED] Set OLLAMA_API_KEY in Colab Secrets to run this cell.')
        return True
    return False


def assert_ok(result, field: str = None):
    """Assert the tool result is not an error dict; optionally check a field exists."""
    assert result is not None, 'Tool returned no result'
    if isinstance(result, dict):
        if result.get('status') == 'error':
            raise AssertionError(f'Tool error: {result.get("message", result)}')
        if field:
            assert field in result, f'Expected field "{field}" missing from result: {result}'
    print('[OK]')


async def _wait_for_server(url: str, timeout: int = 120, interval: float = 1.0) -> bool:
    """Poll the MCP endpoint until it responds or the timeout expires."""
    deadline = time.time() + timeout
    while time.time() < deadline:
        try:
            async with _httpx.AsyncClient() as http:
                await http.get(url, timeout=2.0,
                               headers={'Accept': 'application/json, text/event-stream'})
                return True
        except (_httpx.ConnectError, _httpx.ConnectTimeout, _httpx.ReadTimeout):
            await asyncio.sleep(interval)
    return False


print('Helpers loaded')

Helpers loaded


## 5  Server startup

Starts the Radiant RAG MCP server in a background daemon thread.
Tries `http` then `streamable-http` transport automatically.
Polls the endpoint for up to 90 s before raising `TimeoutError`.

In [5]:
import subprocess, time as _time

# Kill any process already bound to the port so restarts work cleanly
_port = int(os.environ['RADIANT_PORT'])
subprocess.run(['fuser', '-k', f'{_port}/tcp'], capture_output=True)
_time.sleep(1.0)

logging.basicConfig(stream=__import__('sys').stderr, level=logging.WARNING, force=True)

_server_ready   = threading.Event()
_server_error   = None
_transport_used = None


def _run_server():
    """Target for the daemon thread that hosts the MCP server."""
    global _server_error, _transport_used
    try:
        loop = asyncio.new_event_loop()
        asyncio.set_event_loop(loop)
        from radiant_rag_mcp.server import mcp
        print('Package  : radiant_rag_mcp  ok')
        _server_ready.set()
        host = os.environ['RADIANT_HOST']
        port = int(os.environ['RADIANT_PORT'])
        for _t in ['http', 'streamable-http']:
            try:
                _transport_used = _t
                mcp.run(transport=_t, host=host, port=port)
                return
            except Exception as _e:
                if 'Unknown transport' in str(_e) or 'unknown transport' in str(_e).lower():
                    continue
                raise
        raise RuntimeError('No supported HTTP transport found (tried: http, streamable-http)')
    except Exception as exc:
        _server_error = exc
        if not _server_ready.is_set():
            _server_ready.set()


_thread = threading.Thread(target=_run_server, name='radiant-mcp', daemon=True)
_thread.start()

# Wait for the import to complete before polling the HTTP endpoint
if not _server_ready.wait(timeout=30):
    raise TimeoutError('Server thread did not signal ready within 30 s.')
if _server_error:
    raise _server_error

print(f'Waiting for server at {SERVER_URL} ...')
_deadline = time.time() + 90
while time.time() < _deadline:
    if _server_error:
        raise RuntimeError(f'Server thread failed: {_server_error}')
    if asyncio.run(_wait_for_server(SERVER_URL, timeout=5)):
        print(f'Server ready  ->  {SERVER_URL}  (transport: {_transport_used})')
        break
    time.sleep(1)
else:
    raise TimeoutError('Server did not bind within 90 s.')

Package  : radiant_rag_mcp  ok
Waiting for server at http://127.0.0.1:8765/mcp ...


╭──────────────────────────────────────────────────────────────────────────────╮                  
                 │                                                                              │                  
                 │                                                                              │                  
                 │                         ▄▀▀ ▄▀█ █▀▀ ▀█▀ █▀▄▀█ █▀▀ █▀█                        │                  
                 │                         █▀  █▀█ ▄▄█  █  █ ▀ █ █▄▄ █▀▀                        │                  
                 │                                                                              │                  
                 │                                                                              │                  
                 │                                FastMCP 3.2.4                                 │                  
                 │                            https://gofastmcp.com                             │                  
                 │                                                                              │                  
                 │                  🖥  Server:      radiant-rag, 3.2.4                          │                  
                 │                  🚀 Deploy free: https://horizon.prefect.io                  │                  
                 │                                                                              │                  
                 ╰──────────────────────────────────────────────────────────────────────────────╯

2026-04-23 19:53:59,858 - radiant_rag_mcp.app - INFO - Initializing Radiant RAG...
2026-04-23 19:53:59,859 - radiant_rag_mcp.llm.client - INFO - Using new backend configuration system
2026-04-23 19:53:59,866 - radiant_rag_mcp.llm.backends.factory - INFO - Creating LLM backend: type=ollama
2026-04-23 19:53:59,867 - radiant_rag_mcp.llm.backends.llm_backends - INFO - Configured OpenAI-compatible LLM backend: model=gemma3:12b-cloud, base_url=https://ollama.com/v1
2026-04-23 19:53:59,868 - radiant_rag_mcp.llm.backends.factory - INFO - Creating embedding backend: type=local
2026-04-23 19:53:59,869 - radiant_rag_mcp.llm.backends.embedding_backends - INFO - Initializing sentence-transformers embedding backend: sentence-transformers/all-MiniLM-L12-v2
[transformers] The following layers were not sharded: encoder.layer.*.intermediate.dense.weight, embeddings.LayerNorm.weight, embeddings.LayerNorm.bias, encoder.layer.*.attention.self.query.weight, encoder.layer.*.intermediate.dense.bias, embedding

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

2026-04-23 19:54:02,209 - py.warnings - WARNING - /usr/local/lib/python3.12/dist-packages/radiant_rag_mcp/llm/backends/embedding_backends.py:104: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  self._embedding_dim = self._model.get_sentence_embedding_dimension()

2026-04-23 19:54:02,212 - radiant_rag_mcp.utils.cache - INFO - Initialized EmbeddingCache with max_size=10000 (true LRU)
2026-04-23 19:54:02,213 - radiant_rag_mcp.llm.backends.embedding_backends - INFO - Loaded sentence-transformers embedding model: sentence-transformers/all-MiniLM-L12-v2 (dim=384, device=cuda)
2026-04-23 19:54:02,215 - radiant_rag_mcp.llm.backends.factory - INFO - Creating reranking backend: type=local
2026-04-23 19:54:02,216 - radiant_rag_mcp.llm.backends.reranking_backends - INFO - Initializing cross-encoder reranking backend: cross-encoder/ms-marco-MiniLM-L12-v2
[transformers] The following layers were not sharded: bert.encoder.layer.*.attention

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

2026-04-23 19:54:04,146 - radiant_rag_mcp.llm.backends.reranking_backends - INFO - Loaded cross-encoder reranking model: cross-encoder/ms-marco-MiniLM-L12-v2 (device=cuda)
2026-04-23 19:54:04,152 - radiant_rag_mcp.storage.factory - INFO - Creating Chroma vector store
2026-04-23 19:54:04,591 - radiant_rag_mcp.storage.chroma_store - INFO - Initialized ChromaDB store at 'data/chroma_db' with collection 'radiant_docs'
2026-04-23 19:54:04,592 - radiant_rag_mcp.utils.conversation - INFO - ConversationStore: storage backend is 'chroma' — using in-memory storage
2026-04-23 19:54:04,616 - radiant_rag_mcp.utils.metrics_export - INFO - Prometheus metrics exporter initialized
2026-04-23 19:54:04,618 - PlanningAgent - INFO - [4184a377] [enabled=True] Initialized PlanningAgent
2026-04-23 19:54:04,620 - QueryDecompositionAgent - INFO - [5c05cf54] [enabled=True] Initialized QueryDecompositionAgent
2026-04-23 19:54:04,621 - QueryRewriteAgent - INFO - [01c204db] [enabled=True] Initialized QueryRewriteAg

[04/23/26 19:54:04] INFO     Starting MCP server 'radiant-rag' with transport 'http' on            ]8;id=927922;file:///usr/local/lib/python3.12/dist-packages/fastmcp/server/mixins/transport.py\transport.py]8;;\:]8;id=343148;file:///usr/local/lib/python3.12/dist-packages/fastmcp/server/mixins/transport.py#301\301]8;;\
                             http://127.0.0.1:8765/mcp                                                             

INFO:     Started server process [4613]
INFO:     Waiting for application startup.
2026-04-23 19:54:04,715 - mcp.server.streamable_http_manager - INFO - StreamableHTTP session manager started
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8765 (Press CTRL+C to quit)
2026-04-23 19:54:06,121 - mcp.server.streamable_http_manager - INFO - Created new transport with session ID: b2da469794264417a7b6ce3248a8c6e6


INFO:     127.0.0.1:59578 - "GET /mcp HTTP/1.1" 400 Bad Request
Server ready  ->  http://127.0.0.1:8765/mcp  (transport: http)


## 6  Tool list verification

Lists every registered MCP tool and asserts that `ingest_video` is present.
A missing tool means the video sub-package did not install or register correctly.

In [6]:
# Retrieve the tool manifest from the live server
async def _list_tools():
    async with Client(SERVER_URL) as client:
        return await client.list_tools()

tools      = asyncio.run(_list_tools())
registered = {t.name for t in tools}

print(f'{len(tools)} tool(s) registered:')
for t in sorted(tools, key=lambda x: x.name):
    desc = (t.description or '').splitlines()[0]
    print(f'  {t.name:<30}  {desc}')

# ── ingest_video assertion with actionable diagnostic ────────────────────────
if 'ingest_video' in registered:
    print()
    print('[ASSERT OK] ingest_video is registered')
else:
    import importlib.util
    from pathlib import Path

    print()
    print('[FAIL] ingest_video is NOT in the tool list.')
    print()
    print('Diagnostic:')

    # Locate package root
    spec = importlib.util.find_spec('radiant_rag_mcp')
    if spec:
        root = Path(spec.origin).parent
        print(f'  Package root : {root}')

        # Check which video files exist on disk
        video_files = sorted(root.rglob('*video*.py'))
        if video_files:
            print(f'  Video .py files on disk:')
            for vf in video_files:
                has_tool = 'ingest_video' in vf.read_text()
                marker   = '  <-- ingest_video defined here' if has_tool else ''
                print(f'    {vf.relative_to(root)}{marker}')
        else:
            print('  No video .py files found — section 1 fetch may have failed.')

        # Check server.py for video imports
        server_py = root / 'server.py'
        if server_py.exists():
            srv = server_py.read_text()
            video_imports = [ln.strip() for ln in srv.splitlines()
                             if 'video' in ln.lower() and ln.strip().startswith('import')]
            if video_imports:
                print(f'  server.py video imports : {video_imports}')
            else:
                print('  server.py has NO video imports — patch in section 1 may have failed.')
        else:
            print('  server.py not found.')
    else:
        print('  radiant_rag_mcp not importable — check section 1.')

    print()
    print('  ► Re-run section 1, then restart the runtime and re-run sections 1-6.')
    print('  ► If the file containing ingest_video does not exist in the repo yet,')
    print('    the feature branch name is needed to point RAW_BASE at the right ref.')
    print()
    raise AssertionError(
        'ingest_video not registered after fetch/patch. See diagnostic above.'
    )


2026-04-23 19:54:15,661 - mcp.server.streamable_http_manager - INFO - Created new transport with session ID: 355a1155a2624c518167ba371f6b449e


INFO:     127.0.0.1:40440 - "POST /mcp HTTP/1.1" 200 OK


2026-04-23 19:54:15,675 - mcp.client.streamable_http - INFO - Received session ID: 355a1155a2624c518167ba371f6b449e
2026-04-23 19:54:15,677 - mcp.client.streamable_http - INFO - Negotiated protocol version: 2025-11-25


INFO:     127.0.0.1:40456 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:40458 - "GET /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:40460 - "POST /mcp HTTP/1.1" 200 OK


2026-04-23 19:54:15,709 - mcp.server.lowlevel.server - INFO - Processing request of type ListToolsRequest
2026-04-23 19:54:15,726 - mcp.server.streamable_http - INFO - Terminating session: 355a1155a2624c518167ba371f6b449e


INFO:     127.0.0.1:40466 - "DELETE /mcp HTTP/1.1" 200 OK


2026-04-23 19:54:15,731 - mcp.client.streamable_http - INFO - GET stream disconnected, reconnecting in 1000ms...


11 tool(s) registered:
  clear_index                     Delete all indexed documents from the knowledge base.
  get_index_stats                 Return document counts and system health for the knowledge base.
  ingest_documents                Index local files or directories into the knowledge base.
  ingest_url                      Index a URL or GitHub repository into the knowledge base.
  ingest_video                    Ingest one or more videos into the RAG knowledge base.
  query_knowledge                 Answer a question using the full agentic RAG pipeline.
  rebuild_bm25                    Rebuild the BM25 sparse index from the current vector store contents.
  search_knowledge                Retrieve documents from the knowledge base without LLM generation.
  set_ingest_mode                 Set the default ingestion storage mode for this server session.
  simple_query                    Answer a question using a lightweight retrieval-and-synthesis pipeline.
  start_conversatio

## 7  Create synthetic test videos

Generates two 30-second, 640×480 MP4 files entirely in-process:

| File | Content | Audio |
|------|---------|-------|
| `audio_test.mp4` | SMPTE-style colour bars (6 bands) | 1 kHz sine tone, AAC |
| `silent_test.mp4` | Random solid-colour frames per second | None |

The colour bars give `VideoProcessor` meaningful scene-change boundaries.
The silent video exercises the VLM frame-captioning path in section 10.

In [7]:
import cv2
import numpy as np
import subprocess
from pathlib import Path

VIDEO_DIR    = Path('/tmp/radiant_video_test')
VIDEO_DIR.mkdir(exist_ok=True)
AUDIO_TEST   = str(VIDEO_DIR / 'audio_test.mp4')
SILENT_TEST  = str(VIDEO_DIR / 'silent_test.mp4')

FPS = 10
DUR = 30   # seconds
W, H = 640, 480


# --- Colour-bar generator (SMPTE-style, 6 vertical bands) ---
def _colour_bar_frame(frame_idx: int, total_frames: int) -> np.ndarray:
    BAR_COLOURS = [
        (235, 235, 235),  # White
        (  0, 235, 235),  # Yellow (BGR)
        (235, 235,   0),  # Cyan
        (  0, 235,   0),  # Green
        (235,   0, 235),  # Magenta
        (  0,   0, 235),  # Red
    ]
    frame = np.zeros((H, W, 3), dtype=np.uint8)
    bar_w = W // len(BAR_COLOURS)
    bar_h = int(H * 0.75)
    for i, col in enumerate(BAR_COLOURS):
        x0 = i * bar_w
        x1 = x0 + bar_w if i < len(BAR_COLOURS) - 1 else W
        frame[:bar_h, x0:x1] = col
    for x in range(W):
        g = int((x / W) * 235)
        frame[bar_h:, x] = (g, g, g)
    t_sec = frame_idx / FPS
    cv2.putText(frame, f't={t_sec:.1f}s', (12, H - 12),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1)
    px = int((frame_idx / max(1, total_frames - 1)) * (W - 10)) + 5
    cv2.circle(frame, (px, H - 30), 5, (0, 200, 255), -1)
    return frame


def _write_raw_colourbar(path: str) -> None:
    total  = DUR * FPS
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    writer = cv2.VideoWriter(path, fourcc, FPS, (W, H))
    for fi in range(total):
        writer.write(_colour_bar_frame(fi, total))
    writer.release()


# --- Random-colour generator for silent video ---
def _write_raw_random(path: str) -> None:
    rng    = np.random.default_rng(seed=42)
    total  = DUR * FPS
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    writer = cv2.VideoWriter(path, fourcc, FPS, (W, H))
    for fi in range(total):
        if fi % FPS == 0:
            _col = tuple(int(c) for c in rng.integers(30, 220, size=3))
        frame = np.full((H, W, 3), _col, dtype=np.uint8)
        t_sec = fi / FPS
        cv2.putText(frame, f't={t_sec:.1f}s', (12, 40),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)
        writer.write(frame)
    writer.release()


# ---- Build audio_test.mp4 (colour bars + spoken narration) ------------------
# A pure sine tone yields 0 Whisper segments (no speech content).
# We use espeak TTS to generate real spoken words so Whisper can transcribe
# the audio and produce transcript chunks, which is what the pipeline counts
# as audio_sources.
_raw_a = str(VIDEO_DIR / '_raw_audio.mp4')
_wav   = str(VIDEO_DIR / '_speech.wav')

print('Installing espeak (TTS for Whisper-compatible audio) ...')
subprocess.run(['sudo', 'apt-get', 'install', '-y', '-q', 'espeak'],
               capture_output=True)

# Five 6-second narrated scenes matching the colour bar progression
_NARRATION = (
    'Introduction. This is a colour bar test signal used for video calibration. '
    'Methodology. The display shows SMPTE standard colour bars in six bands. '
    'Experiment. Each band represents a primary or secondary colour: '
    'white, yellow, cyan, green, magenta, and red. '
    'Discussion. The bars are used to verify colour accuracy and signal levels. '
    'Conclusion. This completes the thirty second colour bar test sequence.'
)

print('Generating TTS narration with espeak ...')
r = subprocess.run(
    ['espeak', '-s', '140', '-v', 'en', _NARRATION, '-w', _wav],
    capture_output=True
)
if r.returncode != 0 or not Path(_wav).exists():
    # espeak not available — fall back to ffmpeg TTS filter (flite)
    print('  espeak failed, trying ffmpeg flite filter ...')
    r2 = subprocess.run(
        ['ffmpeg', '-y', '-f', 'lavfi',
         '-i', f'flite=text=\'Introduction. Methodology. Experiment. Discussion. Conclusion.\':voice=rms:duration={DUR}',
         _wav],
        capture_output=True
    )
    if r2.returncode != 0:
        # Last resort: generate a WAV with multiple tones at speech-band
        # frequencies that Whisper may partially detect as voiced audio.
        print('  flite not available; generating voiced-frequency audio ...')
        subprocess.run(
            ['ffmpeg', '-y', '-f', 'lavfi',
             '-i', f'aevalsrc=sin(2*PI*200*t)+0.5*sin(2*PI*400*t)+0.3*sin(2*PI*800*t):s=16000:d={DUR}',
             _wav],
            capture_output=True, check=True
        )
    else:
        print('  flite TTS  OK')
else:
    print('  espeak TTS  OK')

print('Generating colour-bar frames ...')
_write_raw_colourbar(_raw_a)

print('Muxing audio_test.mp4 ...')
subprocess.run(
    ['ffmpeg', '-y', '-i', _raw_a, '-i', _wav,
     '-c:v', 'libx264', '-crf', '28',
     '-c:a', 'aac', '-shortest', AUDIO_TEST],
    capture_output=True, check=True
)

# ---- Build silent_test.mp4 (random colours, no audio stream) ----------------
_raw_s = str(VIDEO_DIR / '_raw_silent.mp4')

print('Generating random-colour frames ...')
_write_raw_random(_raw_s)

print('Encoding silent_test.mp4 (no audio stream) ...')
subprocess.run(
    ['ffmpeg', '-y', '-i', _raw_s,
     '-an',
     '-vcodec', 'libx264', '-crf', '28',
     SILENT_TEST],
    capture_output=True, check=True
)

for _f in [_raw_a, _raw_s, _wav]:
    Path(_f).unlink(missing_ok=True)

print()
print('Test videos created:')
for vp in [AUDIO_TEST, SILENT_TEST]:
    p = Path(vp)
    print(f'  {p.name:<25}  {p.stat().st_size // 1024:>6} KB  ->  {vp}')

assert Path(AUDIO_TEST).exists(),  f'audio_test.mp4 not created: {AUDIO_TEST}'
assert Path(SILENT_TEST).exists(), f'silent_test.mp4 not created: {SILENT_TEST}'
print('[ASSERT OK] Both test videos exist on disk')


Installing espeak (TTS for Whisper-compatible audio) ...
Generating TTS narration with espeak ...
  espeak TTS  OK
Generating colour-bar frames ...
Muxing audio_test.mp4 ...
Generating random-colour frames ...
Encoding silent_test.mp4 (no audio stream) ...

Test videos created:
  audio_test.mp4                327 KB  ->  /tmp/radiant_video_test/audio_test.mp4
  silent_test.mp4                53 KB  ->  /tmp/radiant_video_test/silent_test.mp4
[ASSERT OK] Both test videos exist on disk


## 8  `_has_audio()` smoke test

Calls `VideoProcessor._has_audio()` directly (no MCP round-trip) and asserts:
- `audio_test.mp4` → `True`
- `silent_test.mp4` → `False`

This validates that the ffprobe-backed audio detection works before ingestion.

In [8]:
from radiant_rag_mcp.ingestion.video_processor import VideoProcessor
from radiant_rag_mcp.config import VideoProcessorConfig

# Instantiate with default config; _has_audio() only uses ffprobe
cfg = VideoProcessorConfig()
vp  = VideoProcessor(config=cfg)

test_cases = [
    (AUDIO_TEST,  True,  'audio_test.mp4  should have audio'),
    (SILENT_TEST, False, 'silent_test.mp4 should have NO audio'),
]

for path, expected, description in test_cases:
    result = vp._has_audio(path)
    status = 'ok' if result == expected else 'MISMATCH'
    print(f'  {status}  {Path(path).name:<25}  _has_audio={result}  (expected {expected})  {description}')
    # Hard assertion — mismatch means ffprobe or the file is broken
    assert result == expected, (
        f'_has_audio mismatch for {Path(path).name}: '
        f'got {result}, expected {expected}'
    )

print()
print('[ASSERT OK] _has_audio() returns correct values for both test files')

  ok  audio_test.mp4             _has_audio=True  (expected True)  audio_test.mp4  should have audio
  ok  silent_test.mp4            _has_audio=False  (expected False)  silent_test.mp4 should have NO audio

[ASSERT OK] _has_audio() returns correct values for both test files


## 9  `ingest_video` — audio path  *(Whisper transcription)*

Ingests `audio_test.mp4` via the MCP tool.
Because the file has an audio stream, `VideoProcessor` routes it through
Whisper for transcription.  Asserts:
- `result["audio_sources"] == 1`
- `result["silent_sources"] == 0`

In [9]:
%%time
# Ingest the audio video; disable frame captioning to keep runtime short
print('=== ingest_video (audio_test.mp4 — Whisper path) ===')
r = run('ingest_video', {
    'sources':                [AUDIO_TEST],
    'hierarchical':           True,
    'enable_frame_captioning': False,
    'force_frame_analysis':    False,
    'summarize':               False,
})

assert_ok(r, 'sources_processed')

# Verify routing: audio file must land on the Whisper path
assert r.get('audio_sources')  == 1, \
    f'Expected audio_sources=1, got {r.get("audio_sources")}'
assert r.get('silent_sources') == 0, \
    f'Expected silent_sources=0, got {r.get("silent_sources")}'

print(f'\nChunks created  : {r.get("chunks_created")}')
print(f'Audio sources   : {r.get("audio_sources")}')
print(f'Silent sources  : {r.get("silent_sources")}')
print(f'Errors          : {r.get("errors")}')
print('[ASSERT OK] audio_sources==1, silent_sources==0')

2026-04-23 19:54:58,086 - mcp.server.streamable_http_manager - INFO - Created new transport with session ID: 23504dc2dce04e9d9233a8ded3ab7478


=== ingest_video (audio_test.mp4 — Whisper path) ===
INFO:     127.0.0.1:56034 - "POST /mcp HTTP/1.1" 200 OK


2026-04-23 19:54:58,091 - mcp.client.streamable_http - INFO - Received session ID: 23504dc2dce04e9d9233a8ded3ab7478
2026-04-23 19:54:58,093 - mcp.client.streamable_http - INFO - Negotiated protocol version: 2025-11-25


INFO:     127.0.0.1:56046 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:56050 - "GET /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:56064 - "POST /mcp HTTP/1.1" 200 OK


2026-04-23 19:54:58,112 - mcp.server.lowlevel.server - INFO - Processing request of type CallToolRequest
2026-04-23 19:54:58,312 - radiant_rag_mcp.ingestion.video_processor - INFO - Loading faster-whisper model 'base' on cuda (compute_type=int8)
2026-04-23 19:55:00,399 - radiant_rag_mcp.ingestion.video_processor - INFO - faster-whisper model ready.
2026-04-23 19:55:00,943 - faster_whisper - INFO - Processing audio with duration 00:29.954
2026-04-23 19:55:01,429 - faster_whisper - INFO - Detected language 'en' with probability 0.85
2026-04-23 19:55:01,967 - radiant_rag_mcp.ingestion.video_processor - INFO - Transcribed /tmp/radiant_video_test/audio_test.mp4: 5 segments, language=en
2026-04-23 19:55:01,968 - radiant_rag_mcp.ingestion.video_processor - INFO - Produced 1 transcript chunks from 5 segments (source=/tmp/radiant_video_test/audio_test.mp4)
2026-04-23 19:55:02,333 - radiant_rag_mcp.storage.bm25_index - INFO - Syncing BM25 index with store...
2026-04-23 19:55:02,343 - radiant_rag

INFO:     127.0.0.1:56068 - "POST /mcp HTTP/1.1" 200 OK


2026-04-23 19:55:02,386 - mcp.server.lowlevel.server - INFO - Processing request of type ListToolsRequest
2026-04-23 19:55:02,399 - mcp.server.streamable_http - INFO - Terminating session: 23504dc2dce04e9d9233a8ded3ab7478


INFO:     127.0.0.1:56074 - "DELETE /mcp HTTP/1.1" 200 OK


2026-04-23 19:55:02,404 - mcp.client.streamable_http - INFO - GET stream disconnected, reconnecting in 1000ms...


{
  "sources_processed": 1,
  "sources_failed": 0,
  "chunks_created": 1,
  "documents_stored": 2,
  "silent_sources": 0,
  "audio_sources": 1,
  "summaries": {},
  "errors": []
}
[OK]

Chunks created  : 1
Audio sources   : 1
Silent sources  : 0
Errors          : []
[ASSERT OK] audio_sources==1, silent_sources==0
CPU times: user 2.43 s, sys: 882 ms, total: 3.31 s
Wall time: 4.38 s


In [10]:
# Confirm index stats after audio ingestion
print('=== get_index_stats (after audio ingest) ===')
r = run('get_index_stats')
assert_ok(r)

2026-04-23 19:55:11,573 - mcp.server.streamable_http_manager - INFO - Created new transport with session ID: 1661f9e8530d407eaf4702cdc704c209


=== get_index_stats (after audio ingest) ===
INFO:     127.0.0.1:47478 - "POST /mcp HTTP/1.1" 200 OK


2026-04-23 19:55:11,582 - mcp.client.streamable_http - INFO - Received session ID: 1661f9e8530d407eaf4702cdc704c209
2026-04-23 19:55:11,583 - mcp.client.streamable_http - INFO - Negotiated protocol version: 2025-11-25


INFO:     127.0.0.1:47482 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:47488 - "GET /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:47492 - "POST /mcp HTTP/1.1" 200 OK


2026-04-23 19:55:11,602 - mcp.server.lowlevel.server - INFO - Processing request of type CallToolRequest


INFO:     127.0.0.1:47504 - "POST /mcp HTTP/1.1" 200 OK


2026-04-23 19:55:11,618 - mcp.server.lowlevel.server - INFO - Processing request of type ListToolsRequest
2026-04-23 19:55:11,629 - mcp.server.streamable_http - INFO - Terminating session: 1661f9e8530d407eaf4702cdc704c209


INFO:     127.0.0.1:47520 - "DELETE /mcp HTTP/1.1" 200 OK


2026-04-23 19:55:11,635 - mcp.client.streamable_http - INFO - GET stream disconnected, reconnecting in 1000ms...


{
  "health": {
    "redis": true,
    "vector_index": {
      "name": "radiant_docs",
      "backend": "chroma",
      "persist_directory": "./data/chroma_db",
      "document_count": 2,
      "distance_function": "cosine",
      "embedding_dimension": 384,
      "metadata": {
        "hnsw:space": "cosine"
      }
    },
    "bm25_index": {
      "document_count": 1,
      "unique_terms": 43,
      "avg_doc_length": 54.0,
      "k1": 1.5,
      "b": 0.75,
      "dirty": false,
      "needs_rebuild": true,
      "index_path": "data/bm25_index.json.gz",
      "storage_format": "json.gz"
    },
    "conversation": true
  },
  "metrics": {
    "run_count": 0,
    "average_latency_ms": null,
    "average_success_rate": 1.0,
    "history": []
  }
}
[OK]


## 10  `ingest_video` — silent path  *(VLM frame-window analysis)*  `[VLM]`

Ingests `silent_test.mp4`, which has **no audio stream**.
`VideoProcessor` detects the missing audio and routes to
`_process_silent_video()`: samples frames, tiles filmstrips, then captions
each window with the VLM.

Asserts:
- `result["silent_sources"] == 1`
- `result["audio_sources"] == 0`

> **Runtime:** ~2–5 min on T4 GPU (30 s video ÷ 10 s windows = 3 windows).

In [11]:
%%time
# Ingest the silent video — requires VLM captioner for frame-window analysis.
# Skipped automatically when the VLM (Qwen2-VL) is not available.
if not HAS_VLM:
    print('[SKIPPED] VLM not available.')
    print('  Install: pip install "transformers>=4.45.0" qwen-vl-utils')
    print('  Then restart the runtime and re-run sections 1a → 1b → 3 → 5 → 10.')
else:
    print('=== ingest_video (silent_test.mp4 — VLM frame-window path) ===')
    r = run('ingest_video', {
        'sources':                [SILENT_TEST],
        'hierarchical':           True,
        'enable_frame_captioning': False,
        'force_frame_analysis':    False,
        'summarize':               False,
    })

    assert_ok(r, 'sources_processed')

    assert r.get('silent_sources') == 1, \
        f'Expected silent_sources=1, got {r.get("silent_sources")}'
    assert r.get('audio_sources')  == 0, \
        f'Expected audio_sources=0, got {r.get("audio_sources")}'

    print(f'\nChunks created  : {r.get("chunks_created")}')
    print(f'Audio sources   : {r.get("audio_sources")}')
    print(f'Silent sources  : {r.get("silent_sources")}')
    print(f'Errors          : {r.get("errors")}')
    print('[ASSERT OK] silent_sources==1, audio_sources==0')


2026-04-23 19:55:21,826 - mcp.server.streamable_http_manager - INFO - Created new transport with session ID: 30e3e3d8e7ae4eb092990aa26c993334


=== ingest_video (silent_test.mp4 — VLM frame-window path) ===
INFO:     127.0.0.1:34732 - "POST /mcp HTTP/1.1" 200 OK


2026-04-23 19:55:21,831 - mcp.client.streamable_http - INFO - Received session ID: 30e3e3d8e7ae4eb092990aa26c993334
2026-04-23 19:55:21,834 - mcp.client.streamable_http - INFO - Negotiated protocol version: 2025-11-25


INFO:     127.0.0.1:34738 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:34740 - "GET /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:34746 - "POST /mcp HTTP/1.1" 200 OK


2026-04-23 19:55:21,851 - mcp.server.lowlevel.server - INFO - Processing request of type CallToolRequest
2026-04-23 19:55:22,068 - pyscenedetect - INFO - Detecting scenes...
2026-04-23 19:55:22,510 - radiant_rag_mcp.ingestion.video_processor - INFO - VideoProcessor: window 1/6 [0.0s-10.0s]
2026-04-23 19:55:24,396 - radiant_rag_mcp.ingestion.video_processor - INFO - VideoProcessor: window 2/6 [8.0s-18.0s]
2026-04-23 19:55:26,479 - radiant_rag_mcp.ingestion.video_processor - INFO - VideoProcessor: window 3/6 [16.0s-20.0s]
2026-04-23 19:55:28,829 - radiant_rag_mcp.ingestion.video_processor - INFO - VideoProcessor: window 4/6 [18.0s-23.0s]
2026-04-23 19:55:31,045 - radiant_rag_mcp.ingestion.video_processor - INFO - VideoProcessor: window 5/6 [21.0s-26.0s]
2026-04-23 19:55:33,015 - radiant_rag_mcp.ingestion.video_processor - INFO - VideoProcessor: window 6/6 [24.0s-30.0s]
2026-04-23 19:55:35,002 - radiant_rag_mcp.ingestion.video_processor - INFO - Produced 6 frame-window chunks from 6 windo

INFO:     127.0.0.1:47248 - "POST /mcp HTTP/1.1" 200 OK


2026-04-23 19:55:35,215 - mcp.server.lowlevel.server - INFO - Processing request of type ListToolsRequest
2026-04-23 19:55:35,228 - mcp.server.streamable_http - INFO - Terminating session: 30e3e3d8e7ae4eb092990aa26c993334


INFO:     127.0.0.1:47256 - "DELETE /mcp HTTP/1.1" 200 OK


2026-04-23 19:55:35,234 - mcp.client.streamable_http - INFO - GET stream disconnected, reconnecting in 1000ms...


{
  "sources_processed": 1,
  "sources_failed": 0,
  "chunks_created": 6,
  "documents_stored": 15,
  "silent_sources": 1,
  "audio_sources": 0,
  "summaries": {},
  "errors": []
}
[OK]

Chunks created  : 6
Audio sources   : 0
Silent sources  : 1
Errors          : []
[ASSERT OK] silent_sources==1, audio_sources==0
CPU times: user 3.07 s, sys: 167 ms, total: 3.23 s
Wall time: 13.5 s


In [12]:
%%time
# Smoke-test force_frame_analysis flag: route an audio video through the VLM path.
# Requires VLM — skipped if not available.
if not HAS_VLM:
    print('[SKIPPED] VLM not available — skipping force_frame_analysis smoke test.')
else:
    print('=== ingest_video (audio_test.mp4, force_frame_analysis=True) ===')
    r = run('ingest_video', {
        'sources':             [AUDIO_TEST],
        'hierarchical':        False,
        'force_frame_analysis': True,
        'summarize':            False,
    })
    assert_ok(r)
    print(f'Silent sources (forced): {r.get("silent_sources")}  -- expected 1')


2026-04-23 19:55:42,489 - mcp.server.streamable_http_manager - INFO - Created new transport with session ID: 44219b5ffdbd4d7cb9c8a9540d81c2c3


=== ingest_video (audio_test.mp4, force_frame_analysis=True) ===
INFO:     127.0.0.1:46322 - "POST /mcp HTTP/1.1" 200 OK


2026-04-23 19:55:42,496 - mcp.client.streamable_http - INFO - Received session ID: 44219b5ffdbd4d7cb9c8a9540d81c2c3
2026-04-23 19:55:42,498 - mcp.client.streamable_http - INFO - Negotiated protocol version: 2025-11-25


INFO:     127.0.0.1:46332 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:46334 - "GET /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:46340 - "POST /mcp HTTP/1.1" 200 OK


2026-04-23 19:55:42,518 - mcp.server.lowlevel.server - INFO - Processing request of type CallToolRequest
2026-04-23 19:55:42,725 - pyscenedetect - INFO - Detecting scenes...
2026-04-23 19:55:43,138 - radiant_rag_mcp.ingestion.video_processor - INFO - VideoProcessor: window 1/4 [0.0s-10.0s]
2026-04-23 19:55:45,581 - radiant_rag_mcp.ingestion.video_processor - INFO - VideoProcessor: window 2/4 [8.0s-18.0s]
2026-04-23 19:55:47,844 - radiant_rag_mcp.ingestion.video_processor - INFO - VideoProcessor: window 3/4 [16.0s-26.0s]
2026-04-23 19:55:50,132 - radiant_rag_mcp.ingestion.video_processor - INFO - VideoProcessor: window 4/4 [24.0s-30.0s]
2026-04-23 19:55:52,055 - radiant_rag_mcp.ingestion.video_processor - INFO - Produced 4 frame-window chunks from 4 windows (source=/tmp/radiant_video_test/audio_test.mp4)
2026-04-23 19:55:52,112 - radiant_rag_mcp.storage.bm25_index - INFO - Syncing BM25 index with store...
2026-04-23 19:55:52,124 - radiant_rag_mcp.storage.bm25_index - INFO - Saved BM25 i

INFO:     127.0.0.1:54424 - "POST /mcp HTTP/1.1" 200 OK


2026-04-23 19:55:52,139 - mcp.server.lowlevel.server - INFO - Processing request of type ListToolsRequest
2026-04-23 19:55:52,153 - mcp.server.streamable_http - INFO - Terminating session: 44219b5ffdbd4d7cb9c8a9540d81c2c3


INFO:     127.0.0.1:54438 - "DELETE /mcp HTTP/1.1" 200 OK
{
  "sources_processed": 1,
  "sources_failed": 0,
  "chunks_created": 4,
  "documents_stored": 4,
  "silent_sources": 1,
  "audio_sources": 0,
  "summaries": {},
  "errors": []
}
[OK]
Silent sources (forced): 1  -- expected 1
CPU times: user 2.23 s, sys: 106 ms, total: 2.33 s
Wall time: 9.73 s


## 11  `search_knowledge` across video content

Runs hybrid and BM25 searches against the ingested chunks.
Each result row highlights the `content_type` metadata field
(`transcript` vs `frame_window_captions`) so you can see which ingestion
path produced the hit.

In [13]:
%%time
# Hybrid search — combines dense vector + BM25 scores
print('=== search_knowledge: hybrid ===')
r = run('search_knowledge', {
    'query': 'colour bars calibration visual test pattern',
    'mode':  'hybrid',
    'top_k': 6,
})
assert_ok(r)

if isinstance(r, dict) and r.get('results'):
    print()
    print(f'  {"content_type":<24}  {"source":<24}  {"t/window":<10}  preview')
    print('  ' + '-' * 80)
    for hit in r['results'][:6]:
        meta    = hit.get('meta', hit.get('metadata', {}))  # API returns 'meta'
        ctype   = meta.get('content_type', '?')
        ts      = meta.get('start_time', meta.get('window_index', '?'))
        source  = Path(meta.get('source', '?')).name
        preview = hit.get('content', '')[:80].replace('\n', ' ')
        print(f'  {ctype:<24}  {source:<24}  {str(ts):<10}  {preview}...')


2026-04-23 19:55:58,106 - mcp.server.streamable_http_manager - INFO - Created new transport with session ID: 1264a27a40bc47f7bafeef85c6c4330f


=== search_knowledge: hybrid ===
INFO:     127.0.0.1:38946 - "POST /mcp HTTP/1.1" 200 OK


2026-04-23 19:55:58,111 - mcp.client.streamable_http - INFO - Received session ID: 1264a27a40bc47f7bafeef85c6c4330f
2026-04-23 19:55:58,113 - mcp.client.streamable_http - INFO - Negotiated protocol version: 2025-11-25


INFO:     127.0.0.1:38950 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:38960 - "GET /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:38972 - "POST /mcp HTTP/1.1" 200 OK


2026-04-23 19:55:58,131 - mcp.server.lowlevel.server - INFO - Processing request of type CallToolRequest
2026-04-23 19:55:58,134 - DenseRetrievalAgent - INFO - [6d3d02b6] [enabled=True] Initialized DenseRetrievalAgent
2026-04-23 19:55:58,208 - DenseRetrievalAgent - INFO - [c1a09348] [query_length=43 num_results=6 top_k=6] Dense retrieval completed
2026-04-23 19:55:58,209 - DenseRetrievalAgent - INFO - [c1a09348] [duration_ms=73.82 status=success] Execution completed
2026-04-23 19:55:58,209 - BM25RetrievalAgent - INFO - [6218408f] [enabled=True] Initialized BM25RetrievalAgent
2026-04-23 19:55:58,220 - BM25RetrievalAgent - INFO - [505ecae5] [query_length=43 num_results=6 top_k=6] BM25 retrieval completed
2026-04-23 19:55:58,221 - BM25RetrievalAgent - INFO - [505ecae5] [duration_ms=10.03 status=success] Execution completed
2026-04-23 19:55:58,221 - RRFAgent - INFO - [7da93cf4] [enabled=True] Initialized RRFAgent
2026-04-23 19:55:58,223 - RRFAgent - INFO - [32508d5e] [num_runs=2 total_docs

INFO:     127.0.0.1:38986 - "POST /mcp HTTP/1.1" 200 OK


2026-04-23 19:55:58,240 - mcp.server.lowlevel.server - INFO - Processing request of type ListToolsRequest
2026-04-23 19:55:58,252 - mcp.server.streamable_http - INFO - Terminating session: 1264a27a40bc47f7bafeef85c6c4330f


INFO:     127.0.0.1:38988 - "DELETE /mcp HTTP/1.1" 200 OK


2026-04-23 19:55:58,257 - mcp.client.streamable_http - INFO - GET stream disconnected, reconnecting in 1000ms...


{
  "query": "colour bars calibration visual test pattern",
  "mode": "hybrid",
  "results": [
    {
      "doc_id": "983fb6841a203aa47a32f72d024d6ea3c9e1bb23377add65c3640b449afab524",
      "content": "Introduction. This is a color bar test signal used for video calibration. Methodology. The display shows SMT standard color bar in six bands. Experiment. Each band represents a primary or secondary color. White, yellow, cyan, green, magenta, and red. Discussion. The bar are used to verify color accuracy and signal levels. Conclusion. This completes the 30.",
      "meta": {
        "end_time": 30.0,
        "has_embedding": true,
        "content_type": "transcript",
        "video_id": "",
        "language": "en",
        "total_chunks": 1,
        "title": "audio_test",
        "duration": 30.0,
        "doc_level": "child",
        "start_time": 0.0,
        "is_youtube": false,
        "child_index": 0,
        "is_silent": false,
        "child_total": 1,
        "chunk_index": 0,

In [14]:
%%time
# BM25-only search for a term that appears in transcript/caption text
print('=== search_knowledge: bm25 ===')
r = run('search_knowledge', {
    'query': 'frame display colour random pattern',
    'mode':  'bm25',
    'top_k': 5,
})
assert_ok(r)

if isinstance(r, dict) and r.get('results'):
    print()
    for hit in r['results'][:4]:
        meta  = hit.get('meta', hit.get('metadata', {}))  # API returns 'meta'
        ctype = meta.get('content_type', '?')
        score = hit.get('score', '?')
        print(f'  [{ctype}]  score={score:.4f}  {hit.get("content", "")[:90]}')


2026-04-23 19:56:26,103 - mcp.server.streamable_http_manager - INFO - Created new transport with session ID: 1302dbaeedfa45a6a0d19fa6d6289e0e


=== search_knowledge: bm25 ===
INFO:     127.0.0.1:49678 - "POST /mcp HTTP/1.1" 200 OK


2026-04-23 19:56:26,110 - mcp.client.streamable_http - INFO - Received session ID: 1302dbaeedfa45a6a0d19fa6d6289e0e
2026-04-23 19:56:26,113 - mcp.client.streamable_http - INFO - Negotiated protocol version: 2025-11-25


INFO:     127.0.0.1:49682 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:49686 - "GET /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:49696 - "POST /mcp HTTP/1.1" 200 OK


2026-04-23 19:56:26,130 - mcp.server.lowlevel.server - INFO - Processing request of type CallToolRequest
2026-04-23 19:56:26,132 - BM25RetrievalAgent - INFO - [ec730e7e] [enabled=True] Initialized BM25RetrievalAgent
2026-04-23 19:56:26,141 - BM25RetrievalAgent - INFO - [75db6dcb] [query_length=35 num_results=5 top_k=5] BM25 retrieval completed
2026-04-23 19:56:26,142 - BM25RetrievalAgent - INFO - [75db6dcb] [duration_ms=7.94 status=success] Execution completed


INFO:     127.0.0.1:49704 - "POST /mcp HTTP/1.1" 200 OK


2026-04-23 19:56:26,154 - mcp.server.lowlevel.server - INFO - Processing request of type ListToolsRequest
2026-04-23 19:56:26,166 - mcp.server.streamable_http - INFO - Terminating session: 1302dbaeedfa45a6a0d19fa6d6289e0e


INFO:     127.0.0.1:49706 - "DELETE /mcp HTTP/1.1" 200 OK


2026-04-23 19:56:26,172 - mcp.client.streamable_http - INFO - GET stream disconnected, reconnecting in 1000ms...


{
  "query": "frame display colour random pattern",
  "mode": "bm25",
  "results": [
    {
      "doc_id": "b65f26e94952ff3f9ac6b4ec6273344505213011fd68193bea2ba84f7c9f49a5",
      "content": "[8.0s-18.0s] Here's a content analysis of the provided filmstrip stills:\n\nThe video displays a series of vertically oriented color bars characteristic of a test pattern, specifically a SMPTE color bars sequence. The sequence begins with white, yellow, cyan, magenta, and red bars, progressing to green, cyan, yellow, and then blue and yellow bars in the final frame. Underneath each frame is a grayscale ramp, increasing in luminance from left to right, labeled with corresponding timestamps of 8.0s, 13.0s, and 18.0s, indicating the temporal location of each captured frame within the 10.0-second window. The environment appears to be a flat, featureless background providing a consistent display surface for the color reference.",
      "meta": {
        "end_time": 18.0,
        "is_silent": true,
   

## 12  `query_knowledge` grounded in video content  `[LLM]`

> **Requires a configured LLM backend and a valid `OLLAMA_API_KEY`.**

Sends a natural-language question to the RAG pipeline.
The retrieval context is drawn from the ingested video chunks,
exercising both transcript and frame-caption content types.

In [15]:
%%time
if not skip_llm():
    # Query that should pull hits from both the audio (transcript) and silent (captions) video
    print('=== query_knowledge (video-grounded) ===')
    r = run('query_knowledge', {
        'question': (
            'What types of visual content are present in these videos? '
            'Describe any colour patterns, test signals, or visual features '
            'that appear across different time windows.'
        ),
        'mode': 'hybrid',
    })
    assert_ok(r, 'answer')
    print()
    print('Answer:')
    print('-' * 60)
    print(r['answer'][:900])
    print('-' * 60)
    print(f'Warnings : {r.get("warnings", [])}')

2026-04-23 19:56:46,008 - mcp.server.streamable_http_manager - INFO - Created new transport with session ID: 61bff3e0cdd447c7a5ba323fcff63ac8


=== query_knowledge (video-grounded) ===
INFO:     127.0.0.1:38748 - "POST /mcp HTTP/1.1" 200 OK


2026-04-23 19:56:46,015 - mcp.client.streamable_http - INFO - Received session ID: 61bff3e0cdd447c7a5ba323fcff63ac8
2026-04-23 19:56:46,016 - mcp.client.streamable_http - INFO - Negotiated protocol version: 2025-11-25


INFO:     127.0.0.1:38760 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:38768 - "GET /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:38780 - "POST /mcp HTTP/1.1" 200 OK


2026-04-23 19:56:46,038 - mcp.server.lowlevel.server - INFO - Processing request of type CallToolRequest
2026-04-23 19:56:46,041 - radiant_rag_mcp.orchestrator - INFO - [cc3e4ca5-c2b4-4538-aae0-e655a3cea439] Starting agentic pipeline for query: What types of visual content are present in these videos? Describe any colour patterns, test signals...
2026-04-23 19:56:46,042 - radiant_rag_mcp.orchestrator - INFO - [cc3e4ca5-c2b4-4538-aae0-e655a3cea439] Retrieval mode: hybrid, fast_path: False
2026-04-23 19:56:46,096 - radiant_rag_mcp.llm.backends.llm_backends - INFO - Initialized OpenAI-compatible LLM backend: model=gemma3:12b-cloud, base_url=https://ollama.com/v1
2026-04-23 19:56:47,922 - QueryRewriteAgent - INFO - [cc3e4ca5-c2b4-4538-aae0-e655a3cea439] [original=What types of visual content are present in these  rewritten=Analyze videos for visual content types, color pat] Query rewritten
2026-04-23 19:56:47,923 - QueryRewriteAgent - INFO - [cc3e4ca5-c2b4-4538-aae0-e655a3cea439] [duration

INFO:     127.0.0.1:59414 - "POST /mcp HTTP/1.1" 200 OK


2026-04-23 19:56:53,141 - mcp.server.lowlevel.server - INFO - Processing request of type ListToolsRequest
2026-04-23 19:56:53,152 - mcp.server.streamable_http - INFO - Terminating session: 61bff3e0cdd447c7a5ba323fcff63ac8


INFO:     127.0.0.1:59430 - "DELETE /mcp HTTP/1.1" 200 OK


2026-04-23 19:56:53,157 - mcp.client.streamable_http - INFO - GET stream disconnected, reconnecting in 1000ms...


{
  "run_id": "cc3e4ca5-c2b4-4538-aae0-e655a3cea439",
  "answer": "The videos contain various types of visual content, primarily consisting of color transitions and test patterns. Several documents describe sequences of vertically oriented color bars, characteristic of test patterns like SMPTE color bars [DOC 2]. These include white, yellow, cyan, green, magenta, red, and grayscale gradients [DOC 1].\n\nAcross different time windows, the videos exhibit several color patterns:\n\n*   **[8.0s-18.0s]:** A SMPTE color bars sequence, progressing from white, yellow, cyan, magenta, and red to green, cyan, yellow, and blue/yellow [DOC 2]. There's also a grayscale ramp increasing in luminance from left to right [DOC 2]. Later, a light blue transitions to medium purple and then a darker purple [DOC 5].\n*   **[16.0s-20.0s]:** A rapid transition from light beige to deep magenta to bright cyan [DOC 3].\n*   **[21.0s-26.0s]:** A transition from saturated blue to saturated green to saturated peach [

## 13  `VideoSummarizationAgent` — `brief` summary  `[LLM]`

> **Requires a configured LLM backend and VLM.**

`summary_detail='brief'`:
- Window captions: 3–5 focused sentences each
- Chapter summary: 3–5 sentences (1 paragraph)
- Overall summary: 1 short paragraph

In [16]:
%%time
if not skip_llm():
    from radiant_rag_mcp.app import create_app
    from radiant_rag_mcp.agents.video_summarization import VideoSummarizationAgent
    from radiant_rag_mcp.config import VideoSummarizationConfig

    # Build app and extract the shared LLM client once; reused in sections 14-16
    _app = create_app(CONFIG_PATH)
    _llm = _app._orchestrator._llm

    # Retrieve all silent-video chunks from the vector store
    hits   = _app.search('silent_test', mode='dense', top_k=30, show_results=False)
    chunks = [
        doc for doc, _ in hits
        if Path(doc.meta.get('source', '')).name == 'silent_test.mp4'
    ]
    print(f'Chunks retrieved for silent_test.mp4: {len(chunks)}')

    if not chunks:
        print('[NOTE] No chunks found — run section 10 first.')
    else:
        cfg   = VideoSummarizationConfig(summary_detail='brief')
        agent = VideoSummarizationAgent(llm=_llm, config=cfg)
        _res_brief = agent.summarize_video(SILENT_TEST, chunks)

        print(f'Title      : {_res_brief.title}')
        print(f'is_silent  : {_res_brief.is_silent}')
        print(f'Duration   : {_res_brief.duration_seconds:.0f}s')
        print(f'Chapters   : {len(_res_brief.chapters)}')
        print(f'Key topics : {_res_brief.key_topics}')
        print()
        print('Overall summary (brief):')
        print('-' * 60)
        print(_res_brief.summary)
        print('-' * 60)
        print(f'Word count (overall) : {len(_res_brief.summary.split())}')

2026-04-23 19:57:05,293 - radiant_rag_mcp.app - INFO - Initializing Radiant RAG...
2026-04-23 19:57:05,294 - radiant_rag_mcp.llm.client - INFO - Using new backend configuration system
2026-04-23 19:57:05,296 - radiant_rag_mcp.llm.backends.factory - INFO - Creating LLM backend: type=ollama
2026-04-23 19:57:05,297 - radiant_rag_mcp.llm.backends.llm_backends - INFO - Configured OpenAI-compatible LLM backend: model=gemma3:12b-cloud, base_url=https://ollama.com/v1
2026-04-23 19:57:05,298 - radiant_rag_mcp.llm.backends.factory - INFO - Creating embedding backend: type=local
2026-04-23 19:57:05,299 - radiant_rag_mcp.llm.backends.embedding_backends - INFO - Initializing sentence-transformers embedding backend: sentence-transformers/all-MiniLM-L12-v2
[transformers] The following layers were not sharded: encoder.layer.*.intermediate.dense.weight, embeddings.LayerNorm.weight, embeddings.LayerNorm.bias, encoder.layer.*.attention.self.query.weight, encoder.layer.*.intermediate.dense.bias, embedding

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

2026-04-23 19:57:08,452 - py.warnings - WARNING - /usr/local/lib/python3.12/dist-packages/radiant_rag_mcp/llm/backends/embedding_backends.py:104: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  self._embedding_dim = self._model.get_sentence_embedding_dimension()

2026-04-23 19:57:08,453 - radiant_rag_mcp.llm.backends.embedding_backends - INFO - Loaded sentence-transformers embedding model: sentence-transformers/all-MiniLM-L12-v2 (dim=384, device=cuda)
2026-04-23 19:57:08,456 - radiant_rag_mcp.llm.backends.factory - INFO - Creating reranking backend: type=local
2026-04-23 19:57:08,456 - radiant_rag_mcp.llm.backends.reranking_backends - INFO - Initializing cross-encoder reranking backend: cross-encoder/ms-marco-MiniLM-L12-v2
[transformers] The following layers were not sharded: bert.encoder.layer.*.attention.output.LayerNorm.bias, bert.encoder.layer.*.attention.self.key.weight, bert.pooler.dense.weight, bert.encoder.layer.*.ou

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

2026-04-23 19:57:10,560 - radiant_rag_mcp.llm.backends.reranking_backends - INFO - Loaded cross-encoder reranking model: cross-encoder/ms-marco-MiniLM-L12-v2 (device=cuda)
2026-04-23 19:57:10,561 - radiant_rag_mcp.storage.factory - INFO - Creating Chroma vector store
2026-04-23 19:57:10,570 - radiant_rag_mcp.storage.chroma_store - INFO - Initialized ChromaDB store at 'data/chroma_db' with collection 'radiant_docs'
2026-04-23 19:57:10,571 - radiant_rag_mcp.utils.conversation - INFO - ConversationStore: storage backend is 'chroma' — using in-memory storage
2026-04-23 19:57:10,572 - PlanningAgent - INFO - [f55bc998] [enabled=True] Initialized PlanningAgent
2026-04-23 19:57:10,574 - QueryDecompositionAgent - INFO - [deb38f49] [enabled=True] Initialized QueryDecompositionAgent
2026-04-23 19:57:10,576 - QueryRewriteAgent - INFO - [8cc09303] [enabled=True] Initialized QueryRewriteAgent
2026-04-23 19:57:10,577 - QueryExpansionAgent - INFO - [86a2a26b] [enabled=True] Initialized QueryExpansionA

Chunks retrieved for silent_test.mp4: 9


2026-04-23 19:57:16,142 - radiant_rag_mcp.agents.video_summarization - INFO - VideoSummarizationAgent: summarized 'silent_test' → 1 chapters, 7 topics, model=unknown


Title      : silent_test
is_silent  : True
Duration   : 30s
Chapters   : 1
Key topics : ['Color transitions', 'Visual study', 'Color gradients', 'Abstract visuals', 'Smooth shifts', 'Color codes', 'Static background']

Overall summary (brief):
------------------------------------------------------------
This video, titled "silent_test," is a short, 30-second exploration of color transitions designed as a visual study. It lacks any narrative or representational content, focusing entirely on the shifting hues across a static background.

The video’s content consists solely of a sequence of continuous, smooth color gradients. Starting with yellow transitioning to navy blue, it moves through a series of changes including light blue to dark purple, then a rapid cycle of light beige, deep magenta, and bright cyan. Further color evolutions include deep purple to light blue to bright green, blue to green to peach, and concluding with deep magenta subtly brightening to a reddish-magenta. Specif

## 14  `VideoSummarizationAgent` — `standard` summary  `[LLM]`

> **Requires a configured LLM backend and VLM.**

`summary_detail='standard'`:
- Chapter summary: 1–2 structured paragraphs (5–10 sentences)
- Overall summary: 2–3 paragraphs (Overview + Content + optional Detail)

In [17]:
%%time
if not skip_llm():
    cfg   = VideoSummarizationConfig(summary_detail='standard')
    agent = VideoSummarizationAgent(llm=_llm, config=cfg)
    _res_standard = agent.summarize_video(SILENT_TEST, chunks)

    print('Overall summary (standard):')
    print('-' * 60)
    print(_res_standard.summary)
    print('-' * 60)
    print(f'Word count (overall) : {len(_res_standard.summary.split())}')
    print()
    print('Chapter breakdowns:')
    for ch in _res_standard.chapters:
        print(f'  Ch {ch.index + 1}: [{ch.start:.0f}s – {ch.end:.0f}s]  {ch.title}')
        for ln in ch.summary.split('\n')[:4]:
            print(f'    {ln[:110]}')
        print()

2026-04-23 19:57:28,434 - VideoSummarizationAgent - INFO - [ab363288] [enabled=True] Initialized VideoSummarizationAgent
2026-04-23 19:57:28,435 - radiant_rag_mcp.agents.video_summarization - INFO - VideoSummarizationAgent: 9 chunks → 1 chapters (source=/tmp/radiant_video_test/silent_test.mp4, type=frame_window_captions)
2026-04-23 19:57:34,489 - radiant_rag_mcp.agents.video_summarization - INFO - VideoSummarizationAgent: summarized 'silent_test' → 1 chapters, 7 topics, model=unknown


Overall summary (standard):
------------------------------------------------------------
This video, titled "silent_test," is a short, 30-second abstract visual exploration focused entirely on color transitions. It serves as a purely visual study, devoid of any narrative, objects, or text, concentrating solely on the movement and blending of colors across the screen.

The video presents a sequence of gradual color shifts. Initially, the screen displays a smooth transition from yellow to navy blue. This is followed by numerous other transitions, including light blue to deep purple, beige to magenta to cyan, deep purple to light blue to bright green, saturated blue to green, and peach, finally concluding with deep magenta fading to reddish-magenta. The transitions are smooth and continuous, with a variety of colors—yellow, blue, purple, beige, magenta, cyan, green, and peach—being showcased, and specific hex codes noted for some of the color shifts.
--------------------------------------

## 15  `VideoSummarizationAgent` — `detailed` summary  `[LLM]`

> **Requires a configured LLM backend and VLM.**

`summary_detail='detailed'`:
- Chapter summary: 2–3 paragraphs (8–15 sentences)
- Overall summary: 3–4 structured paragraphs
  (Overview / Content / Detail / Takeaway schema)

In [18]:
%%time
if not skip_llm():
    cfg   = VideoSummarizationConfig(summary_detail='detailed')
    agent = VideoSummarizationAgent(llm=_llm, config=cfg)
    _res_detailed = agent.summarize_video(SILENT_TEST, chunks)

    print('Overall summary (detailed):')
    print('-' * 60)
    print(_res_detailed.summary)
    print('-' * 60)
    print(f'Word count (overall) : {len(_res_detailed.summary.split())}')
    print()
    print(f'Chapters : {len(_res_detailed.chapters)}')
    for ch in _res_detailed.chapters:
        print(f'  Ch {ch.index + 1}: [{ch.start:.0f}s – {ch.end:.0f}s]  {ch.title}')

2026-04-23 19:57:49,331 - VideoSummarizationAgent - INFO - [dbb7abc2] [enabled=True] Initialized VideoSummarizationAgent
2026-04-23 19:57:49,333 - radiant_rag_mcp.agents.video_summarization - INFO - VideoSummarizationAgent: 9 chunks → 1 chapters (source=/tmp/radiant_video_test/silent_test.mp4, type=frame_window_captions)
2026-04-23 19:57:55,322 - radiant_rag_mcp.agents.video_summarization - INFO - VideoSummarizationAgent: summarized 'silent_test' → 1 chapters, 6 topics, model=unknown


Overall summary (detailed):
------------------------------------------------------------
This video, titled "silent_test," is a short, abstract visual piece exploring color transitions. Lasting 30 seconds, it functions as a purely aesthetic demonstration, devoid of any narrative elements, objects, or text. The video's primary subject is the seamless and controlled shifting of colors across a static background.

The video's content consists entirely of a series of gradual color changes. Beginning with a yellow-to-navy blue transition, the sequence then progresses through a series of solid color fields morphing into one another – light blue to purple, beige to magenta to cyan, deep purple to light blue to bright green, blue to green to peach, and finally deep magenta to reddish-magenta. Color values are occasionally indicated, suggesting a deliberate and precise control over the color palette and rendering.



This video presents a pattern of gradual, continuous color shifts across a sta

## 16  Summary detail levels — word-count comparison  `[LLM]`

> **Requires a configured LLM backend and VLM.**

Runs all three detail levels and prints a side-by-side table covering:
- **Window** average word count (across all window captions, where available)
- **Chapter** average word count
- **Overall** total word count

Expected trend: `brief` < `standard` < `detailed` in every column.

In [19]:
%%time
if not skip_llm():
    # Collect results for all three detail levels
    # Reuse cached results from sections 13-15 if they exist; otherwise re-run
    detail_results = {}
    for detail, cached in [
        ('brief',    globals().get('_res_brief')),
        ('standard', globals().get('_res_standard')),
        ('detailed', globals().get('_res_detailed')),
    ]:
        if cached is not None:
            detail_results[detail] = cached
            print(f'  {detail:<10}  (using cached result from section 13/14/15)')
        else:
            cfg   = VideoSummarizationConfig(summary_detail=detail)
            agent = VideoSummarizationAgent(llm=_llm, config=cfg)
            detail_results[detail] = agent.summarize_video(SILENT_TEST, chunks)
            print(f'  {detail:<10}  (freshly generated)')

    print()
    # Build comparison table: window / chapter / overall word counts
    HDR = f'  {"Detail":<10}  {"Window avg":>12}  {"Chapter avg":>12}  {"Overall total":>14}  {"Chapters":>8}'
    SEP = '  ' + '-' * (len(HDR) - 2)
    print(HDR)
    print(SEP)

    for detail, res in detail_results.items():
        # Overall summary word count
        overall_wc = len(res.summary.split())

        # Chapter average word count
        ch_wcs = [len(ch.summary.split()) for ch in res.chapters]
        ch_avg = (sum(ch_wcs) // max(1, len(ch_wcs))) if ch_wcs else 0

        # Window average word count — use window_captions if present, else chapter captions
        win_wcs = []
        if hasattr(res, 'window_captions') and res.window_captions:
            win_wcs = [len(w.caption.split()) for w in res.window_captions if hasattr(w, 'caption')]
        elif res.chapters:
            # Fall back: treat chapter summaries as window proxies
            win_wcs = ch_wcs
        win_avg = (sum(win_wcs) // max(1, len(win_wcs))) if win_wcs else 0

        print(f'  {detail:<10}  {win_avg:>12,}  {ch_avg:>12,}  {overall_wc:>14,}  {len(res.chapters):>8}')

    print()
    print('[NOTE] Window avg falls back to chapter avg when per-window captions are unavailable.')

  brief       (using cached result from section 13/14/15)
  standard    (using cached result from section 13/14/15)
  detailed    (using cached result from section 13/14/15)

  Detail        Window avg   Chapter avg   Overall total  Chapters
  ----------------------------------------------------------------
  brief                175           175             122         1
  standard             191           191             134         1
  detailed             201           201             151         1

[NOTE] Window avg falls back to chapter avg when per-window captions are unavailable.
CPU times: user 486 µs, sys: 0 ns, total: 486 µs
Wall time: 480 µs


## 17  Remote URL ingestion (YouTube + any yt-dlp source)  `[LLM]`

> **Requires a configured LLM backend and VLM.**

Tests `ingest_video` with remote URLs.  yt-dlp is the download backend, so
**any URL supported by yt-dlp works** — YouTube, Twitter/X, TikTok, Instagram,
Vimeo, Twitch, Reddit, and 1 000+ other platforms.

Two sub-cells are provided:

| Sub-cell | URL | Path |
|----------|-----|------|
| 17a | YouTube (3Blue1Brown — neural networks) | Whisper audio |
| 17b | Twitter/X public video | Whisper audio (or VLM if silent) |

Search is verified in 17c against whichever chunks were indexed.

> **Runtime:** ~60–120 s per URL (yt-dlp download + Whisper base).


In [20]:
%%time
if not skip_llm():
    YT_URL = 'https://www.youtube.com/watch?v=aircAruvnKk'

    # Any http:// or https:// URL is routed through yt-dlp automatically.
    # YouTube, Twitter/X, TikTok, Vimeo, etc. all use the same code path.
    print(f'=== ingest_video (YouTube URL) ===')
    print(f'URL: {YT_URL}')
    r = run('ingest_video', {
        'sources':    [YT_URL],
        'hierarchical': True,
        'summarize':    True,
    })
    assert_ok(r, 'sources_processed')

    print(f'\nChunks created : {r.get("chunks_created")}')
    print(f'Audio sources  : {r.get("audio_sources")}')
    print(f'Silent sources : {r.get("silent_sources")}')
    print(f'Errors         : {r.get("errors")}')

    for src, s in r.get('summaries', {}).items():
        print(f'\n--- Summary for: {src} ---')
        print(f'  Title     : {s.get("title")}')
        print(f'  is_silent : {s.get("is_silent")}')
        print(f'  Chapters  : {len(s.get("chapters", []))}')
        print()
        print(s.get('summary', '')[:700])


2026-04-23 19:58:17,695 - mcp.server.streamable_http_manager - INFO - Created new transport with session ID: a992e5415d524447aa38952959effe2b


=== ingest_video (YouTube URL) ===
URL: https://www.youtube.com/watch?v=aircAruvnKk
INFO:     127.0.0.1:38094 - "POST /mcp HTTP/1.1" 200 OK


2026-04-23 19:58:17,702 - mcp.client.streamable_http - INFO - Received session ID: a992e5415d524447aa38952959effe2b
2026-04-23 19:58:17,704 - mcp.client.streamable_http - INFO - Negotiated protocol version: 2025-11-25


INFO:     127.0.0.1:38102 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:38112 - "GET /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:38126 - "POST /mcp HTTP/1.1" 200 OK


2026-04-23 19:58:17,728 - mcp.server.lowlevel.server - INFO - Processing request of type CallToolRequest


...

INFO:     127.0.0.1:38270 - "POST /mcp HTTP/1.1" 200 OK


2026-04-23 19:59:13,941 - mcp.server.lowlevel.server - INFO - Processing request of type ListToolsRequest
2026-04-23 19:59:13,955 - mcp.server.streamable_http - INFO - Terminating session: a992e5415d524447aa38952959effe2b


INFO:     127.0.0.1:38274 - "DELETE /mcp HTTP/1.1" 200 OK


2026-04-23 19:59:13,960 - mcp.client.streamable_http - INFO - GET stream disconnected, reconnecting in 1000ms...


{
  "sources_processed": 1,
  "sources_failed": 0,
  "chunks_created": 23,
  "documents_stored": 89,
  "silent_sources": 0,
  "audio_sources": 1,
  "summaries": {
    "https://www.youtube.com/watch?v=aircAruvnKk": {
      "source": "https://www.youtube.com/watch?v=aircAruvnKk",
      "title": "But what is a neural network? | Deep learning chapter 1",
      "duration_seconds": 1120.0,
      "language": "en",
      "is_silent": false,
      "summary": "## Overall Summary: But What is a Neural Network? | Deep Learning Chapter 1\n\nThis video, lasting approximately 11 minutes, serves as an introductory exploration of neural networks, specifically aiming to demystify their structure and function without requiring prior technical expertise. It tackles the challenging problem of replicating human visual processing \u2013 recognizing handwritten digits \u2013 using a computer program and lays the groundwork for a series focused on deep learning. The video\u2019s goal is to explain the foundati

In [21]:
%%time
# Twitter/X URL test — demonstrates non-YouTube remote URL ingestion.
# Replace TWITTER_URL with any public Twitter/X video post.
# The pipeline is identical: yt-dlp downloads, then Whisper or VLM processes.
if not skip_llm():
    TWITTER_URL = 'https://twitter.com/i/status/2047129605996192012'

    print(f'=== ingest_video (Twitter/X URL) ===')
    print(f'URL: {TWITTER_URL}')
    r = run('ingest_video', {
        'sources':    [TWITTER_URL],
        'hierarchical': True,
        'summarize':    True,
    })
    assert_ok(r, 'sources_processed')

    print(f'\nChunks created : {r.get("chunks_created")}')
    print(f'Audio sources  : {r.get("audio_sources")}')
    print(f'Silent sources : {r.get("silent_sources")}')
    print(f'Errors         : {r.get("errors")}')

    for src, s in r.get('summaries', {}).items():
        print(f'\n--- Summary for: {src} ---')
        print(f'  Title     : {s.get("title")}')
        print(f'  is_silent : {s.get("is_silent")}')
        print(f'  Chapters  : {len(s.get("chapters", []))}')
        print()
        print(s.get('summary', '')[:700])


2026-04-23 19:59:29,742 - mcp.server.streamable_http_manager - INFO - Created new transport with session ID: ac25e5783f7b40cfae29925d6ce30da4


=== ingest_video (Twitter/X URL) ===
URL: https://twitter.com/i/status/2047129605996192012
INFO:     127.0.0.1:40030 - "POST /mcp HTTP/1.1" 200 OK


2026-04-23 19:59:29,748 - mcp.client.streamable_http - INFO - Received session ID: ac25e5783f7b40cfae29925d6ce30da4
2026-04-23 19:59:29,749 - mcp.client.streamable_http - INFO - Negotiated protocol version: 2025-11-25


INFO:     127.0.0.1:40040 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:40056 - "GET /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:40060 - "POST /mcp HTTP/1.1" 200 OK


2026-04-23 19:59:29,768 - mcp.server.lowlevel.server - INFO - Processing request of type CallToolRequest


...

INFO:     127.0.0.1:51154 - "POST /mcp HTTP/1.1" 200 OK


2026-04-23 19:59:40,359 - mcp.server.lowlevel.server - INFO - Processing request of type ListToolsRequest
2026-04-23 19:59:40,371 - mcp.server.streamable_http - INFO - Terminating session: ac25e5783f7b40cfae29925d6ce30da4


INFO:     127.0.0.1:51166 - "DELETE /mcp HTTP/1.1" 200 OK


2026-04-23 19:59:40,376 - mcp.client.streamable_http - INFO - GET stream disconnected, reconnecting in 1000ms...


{
  "sources_processed": 1,
  "sources_failed": 0,
  "chunks_created": 1,
  "documents_stored": 3,
  "silent_sources": 0,
  "audio_sources": 1,
  "summaries": {
    "https://twitter.com/i/status/2047129605996192012": {
      "source": "https://twitter.com/i/status/2047129605996192012",
      "title": "Rapid Response 47 - .@FBIDirectorKash: The charity that supposedly fought the Klan funded...",
      "duration_seconds": 56.823,
      "language": "en",
      "is_silent": false,
      "summary": "This video, \"Rapid Response 47 - .@FBIDirectorKash: The charity that supposedly fought the Klan funded...\", is a short commentary (57 seconds) focused on allegations of financial misconduct and hypocrisy surrounding the Southern Poverty Law Center (SPLC). The primary subject is the claim that the SPLC engaged in fraudulent activity to funnel donor money and secretly fund hate groups.\n\nThe video presents a central accusation: the SPLC allegedly defrauded donors of $3 million by employing shel

In [22]:
%%time
if not skip_llm():
    # Search transcript chunks from both remote URL ingests above
    print('=== search_knowledge (remote URL transcripts) ===')
    r = run('search_knowledge', {
        'query': 'neural network layers neurons activation',
        'mode':  'hybrid',
        'top_k': 4,
    })
    assert_ok(r)
    if isinstance(r, dict) and r.get('results'):
        for hit in r['results'][:3]:
            meta  = hit.get('meta', hit.get('metadata', {}))
            ctype = meta.get('content_type', '?')
            src   = Path(meta.get('source', '?')).name[:40]
            print(f'  [{ctype}]  {src}')
            print(f'    {hit.get("content", "")[:100]}')


2026-04-23 20:00:10,431 - mcp.server.streamable_http_manager - INFO - Created new transport with session ID: 7e7065a52bcd47cebbbf38b60c27b160


=== search_knowledge (remote URL transcripts) ===
INFO:     127.0.0.1:52468 - "POST /mcp HTTP/1.1" 200 OK


2026-04-23 20:00:10,437 - mcp.client.streamable_http - INFO - Received session ID: 7e7065a52bcd47cebbbf38b60c27b160
2026-04-23 20:00:10,441 - mcp.client.streamable_http - INFO - Negotiated protocol version: 2025-11-25


INFO:     127.0.0.1:52474 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:52478 - "GET /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:52486 - "POST /mcp HTTP/1.1" 200 OK


2026-04-23 20:00:10,463 - mcp.server.lowlevel.server - INFO - Processing request of type CallToolRequest
2026-04-23 20:00:10,465 - DenseRetrievalAgent - INFO - [cd431784] [enabled=True] Initialized DenseRetrievalAgent
2026-04-23 20:00:10,493 - DenseRetrievalAgent - INFO - [961c86bd] [query_length=40 num_results=4 top_k=4] Dense retrieval completed
2026-04-23 20:00:10,494 - DenseRetrievalAgent - INFO - [961c86bd] [duration_ms=27.76 status=success] Execution completed
2026-04-23 20:00:10,495 - BM25RetrievalAgent - INFO - [f508dc1a] [enabled=True] Initialized BM25RetrievalAgent
2026-04-23 20:00:10,504 - BM25RetrievalAgent - INFO - [d334ee38] [query_length=40 num_results=4 top_k=4] BM25 retrieval completed
2026-04-23 20:00:10,506 - BM25RetrievalAgent - INFO - [d334ee38] [duration_ms=9.26 status=success] Execution completed
2026-04-23 20:00:10,507 - RRFAgent - INFO - [0c2623f2] [enabled=True] Initialized RRFAgent
2026-04-23 20:00:10,509 - RRFAgent - INFO - [5f60876d] [num_runs=2 total_docs=

INFO:     127.0.0.1:52488 - "POST /mcp HTTP/1.1" 200 OK


2026-04-23 20:00:10,529 - mcp.server.lowlevel.server - INFO - Processing request of type ListToolsRequest
2026-04-23 20:00:10,541 - mcp.server.streamable_http - INFO - Terminating session: 7e7065a52bcd47cebbbf38b60c27b160


INFO:     127.0.0.1:52500 - "DELETE /mcp HTTP/1.1" 200 OK


2026-04-23 20:00:10,547 - mcp.client.streamable_http - INFO - GET stream disconnected, reconnecting in 1000ms...


{
  "query": "neural network layers neurons activation",
  "mode": "hybrid",
  "results": [
    {
      "doc_id": "f499441271230c91185000c9720e51f64489c0b4a5aa35c07e9a8b0e63c470e0",
      "content": "Each one of these holds a number that represents the gray scale value of the corresponding pixel, ranging from zero for black pixels up to one for white pixels. This number inside the neuron is called its activation, and the image you might have in mind here is that each neuron is lit up when its activation is a high number. So all of these 784 neurons make up the first layer of our network. Now jumping over to the last layer, this has 10 neurons, each representing one of the digits. The activation in",
      "meta": {
        "chunk_index": 4,
        "language": "en",
        "source": "https://www.youtube.com/watch?v=aircAruvnKk",
        "content_type": "transcript",
        "start_time": 200.0,
        "child_index": 0,
        "has_embedding": true,
        "end_time": 260.0,
       

In [24]:
%%time
# Twitter/X URL test — demonstrates non-YouTube remote URL ingestion.
# Replace TWITTER_URL with any public Twitter/X video post.
# The pipeline is identical: yt-dlp downloads, then Whisper or VLM processes.
if not skip_llm():
    # TWITTER_URL = 'https://twitter.com/i/status/2047129605996192012'
    TWITTER_URL = 'https://www.foxnews.com/video/6393717417112'

    print(f'=== ingest_video (Twitter/X URL) ===')
    print(f'URL: {TWITTER_URL}')
    r = run('ingest_video', {
        'sources':    [TWITTER_URL],
        'hierarchical': True,
        'summarize':    True,
    })
    assert_ok(r, 'sources_processed')

    print(f'\nChunks created : {r.get("chunks_created")}')
    print(f'Audio sources  : {r.get("audio_sources")}')
    print(f'Silent sources : {r.get("silent_sources")}')
    print(f'Errors         : {r.get("errors")}')

    for src, s in r.get('summaries', {}).items():
        print(f'\n--- Summary for: {src} ---')
        print(f'  Title     : {s.get("title")}')
        print(f'  is_silent : {s.get("is_silent")}')
        print(f'  Chapters  : {len(s.get("chapters", []))}')
        print()
        print(s.get('summary', '')[:700])

2026-04-23 20:08:05,673 - mcp.server.streamable_http_manager - INFO - Created new transport with session ID: 94ebeee57833444fa94bb21bb3e7b65d


=== ingest_video (Twitter/X URL) ===
URL: https://www.foxnews.com/video/6393717417112
INFO:     127.0.0.1:55038 - "POST /mcp HTTP/1.1" 200 OK


2026-04-23 20:08:05,680 - mcp.client.streamable_http - INFO - Received session ID: 94ebeee57833444fa94bb21bb3e7b65d
2026-04-23 20:08:05,681 - mcp.client.streamable_http - INFO - Negotiated protocol version: 2025-11-25


INFO:     127.0.0.1:55042 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:55058 - "GET /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:55064 - "POST /mcp HTTP/1.1" 200 OK


2026-04-23 20:08:05,701 - mcp.server.lowlevel.server - INFO - Processing request of type CallToolRequest


2026-04-23 20:08:06,950 - radiant_rag_mcp.ingestion.video_processor - INFO - Remote download complete: 'NYC Council Member Chi Ossé arrested in Brooklyn' (6393717417112, 60s, silent=False, youtube=False)
2026-04-23 20:08:06,951 - radiant_rag_mcp.ingestion.video_processor - INFO - Loading faster-whisper model 'base' on cuda (compute_type=int8)
2026-04-23 20:08:07,791 - radiant_rag_mcp.ingestion.video_processor - INFO - faster-whisper model ready.
2026-04-23 20:08:08,453 - faster_whisper - INFO - Processing audio with duration 01:00.032
2026-04-23 20:08:08,528 - faster_whisper - INFO - Detected language 'en' with probability 0.70
2026-04-23 20:08:20,728 - radiant_rag_mcp.ingestion.video_processor - INFO - Transcribed /tmp/radiant_video_djl5nb0a/6393717417112.mp4: 10 segments, language=en
2026-04-23 20:08:20,729 - radiant_rag_mcp.ingestion.video_processor - INFO - Produced 1 transcript chunks from 10 segments (source=https://www.foxnews.com/video/6393717417112)
2026-04-23 20:08:20,806 - V

INFO:     127.0.0.1:42156 - "POST /mcp HTTP/1.1" 200 OK


2026-04-23 20:08:26,001 - mcp.server.lowlevel.server - INFO - Processing request of type ListToolsRequest
2026-04-23 20:08:26,014 - mcp.server.streamable_http - INFO - Terminating session: 94ebeee57833444fa94bb21bb3e7b65d


INFO:     127.0.0.1:42164 - "DELETE /mcp HTTP/1.1" 200 OK
{
  "sources_processed": 1,
  "sources_failed": 0,
  "chunks_created": 1,
  "documents_stored": 4,
  "silent_sources": 0,
  "audio_sources": 1,
  "summaries": {
    "https://www.foxnews.com/video/6393717417112": {
      "source": "https://www.foxnews.com/video/6393717417112",
      "title": "NYC Council Member Chi Oss\u00e9 arrested in Brooklyn",
      "duration_seconds": 60.0,
      "language": "en",
      "is_silent": false,
      "summary": "This video is a short, 60-second clip documenting a tense and confrontational interaction, seemingly capturing a moment of arrest. The primary subject appears to be New York City Council Member Chi Oss\u00e9, though his identity is initially obscured within the aggressive questioning. The video's topic is the arrest itself, but it focuses heavily on the unsettling and hostile nature of the immediate questioning surrounding the event.\n\nThe video\u2019s content unfolds rapidly and without

## 18  Cleanup

- Clears the ChromaDB vector index
- Deletes the temporary MP4 files from `/tmp/radiant_video_test/`

Skip the `clear_index` call if you want to keep the chunks for further
experimentation; set `RUN_CLEAR = False` in the cell below.

In [25]:
RUN_CLEAR = True   # Set to False to keep the index intact

if RUN_CLEAR:
    print('=== clear_index ===')
    r = run('clear_index', {'confirm': True})
    if isinstance(r, dict):
        if r.get('cleared'):
            print('[OK] Index cleared.')
        elif ('_ensure_index' in r.get('message', '')
              or 'clear failed' in r.get('message', '').lower()):
            # Known ChromaDB reinit issue — collection was dropped
            print('[OK] Collection dropped (known ChromaDB reinit pattern).')
        else:
            raise AssertionError(f'Unexpected clear failure: {r}')

    # Confirm document count is 0
    stats = asyncio.run(_call('get_index_stats'))
    if isinstance(stats, dict):
        vi = stats.get('health', stats).get('vector_index', {})
        print(f'Vector doc count post-clear: {vi.get("document_count", "?")}')
else:
    print('RUN_CLEAR=False — index preserved.')

2026-04-23 20:13:35,061 - mcp.server.streamable_http_manager - INFO - Created new transport with session ID: d3ec53fadddf4119b2e6bd5f23756988


=== clear_index ===
INFO:     127.0.0.1:47856 - "POST /mcp HTTP/1.1" 200 OK


2026-04-23 20:13:35,066 - mcp.client.streamable_http - INFO - Received session ID: d3ec53fadddf4119b2e6bd5f23756988
2026-04-23 20:13:35,067 - mcp.client.streamable_http - INFO - Negotiated protocol version: 2025-11-25


INFO:     127.0.0.1:47860 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:47868 - "GET /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:47872 - "POST /mcp HTTP/1.1" 200 OK


2026-04-23 20:13:35,086 - mcp.server.lowlevel.server - INFO - Processing request of type CallToolRequest
2026-04-23 20:13:35,124 - radiant_rag_mcp.storage.chroma_store - INFO - Dropped Chroma collection 'radiant_docs'
2026-04-23 20:13:35,125 - radiant_rag_mcp.app - ERROR - Failed to clear index: 'ChromaVectorStore' object has no attribute '_ensure_index'


...

INFO:     127.0.0.1:47914 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:47928 - "GET /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:47942 - "POST /mcp HTTP/1.1" 200 OK


2026-04-23 20:13:35,229 - mcp.server.lowlevel.server - INFO - Processing request of type CallToolRequest


INFO:     127.0.0.1:47956 - "POST /mcp HTTP/1.1" 200 OK


2026-04-23 20:13:35,245 - mcp.server.lowlevel.server - INFO - Processing request of type ListToolsRequest
2026-04-23 20:13:35,255 - mcp.server.streamable_http - INFO - Terminating session: 9ef51cd2190f4934ad445a84007b4e23


INFO:     127.0.0.1:47972 - "DELETE /mcp HTTP/1.1" 200 OK
Vector doc count post-clear: 0


In [26]:
import shutil

# Delete temporary video files
_test_dir = Path('/tmp/radiant_video_test')
if _test_dir.exists():
    shutil.rmtree(_test_dir)
    print(f'Removed test video directory: {_test_dir}')
else:
    print(f'Directory not found (already cleaned?): {_test_dir}')

print('Cleanup complete')

Removed test video directory: /tmp/radiant_video_test
Cleanup complete


## Summary

Run this cell at any point for a status overview.

In [27]:
print('=' * 65)
print('  Radiant RAG MCP — Video Ingestion & Summarization Test')
print('  Run complete')
print('=' * 65)
print()
print('  Synthetic test videos  (640x480, 30 s)')
print('    audio_test.mp4   — SMPTE colour bars + espeak TTS narration')
print('    silent_test.mp4  — random solid-colour frames, no audio')
print()
print('  Audio detection')
print('    ok  _has_audio(audio_test.mp4)  == True')
print('    ok  _has_audio(silent_test.mp4) == False')
print()
print('  Video ingestion')
print('    ok  audio_test.mp4   -> Whisper transcription -> transcript chunks')
print('    ok  silent_test.mp4  -> VLM frame windows     -> caption chunks')
print('    ok  force_frame_analysis flag override')
print()
print('  Search')
print('    ok  search_knowledge hybrid  (content_type highlighted)')
print('    ok  search_knowledge bm25')
print()
print('  LLM / VLM cells  [L = requires OLLAMA_API_KEY]')
print('   [L]  query_knowledge grounded in video content')
print('   [L]  VideoSummarizationAgent  brief')
print('   [L]  VideoSummarizationAgent  standard')
print('   [L]  VideoSummarizationAgent  detailed')
print('   [L]  Word-count table  (window / chapter / overall)')
print('   [L]  YouTube URL ingestion + transcript search')
print()
print(f'  LLM key available : {HAS_LLM_KEY}')
print(f'  VLM available     : {HAS_VLM}')


  Radiant RAG MCP — Video Ingestion & Summarization Test
  Run complete

  Synthetic test videos  (640x480, 30 s)
    audio_test.mp4   — SMPTE colour bars + espeak TTS narration
    silent_test.mp4  — random solid-colour frames, no audio

  Audio detection
    ok  _has_audio(audio_test.mp4)  == True
    ok  _has_audio(silent_test.mp4) == False

  Video ingestion
    ok  audio_test.mp4   -> Whisper transcription -> transcript chunks
    ok  silent_test.mp4  -> VLM frame windows     -> caption chunks
    ok  force_frame_analysis flag override

  Search
    ok  search_knowledge hybrid  (content_type highlighted)
    ok  search_knowledge bm25

  LLM / VLM cells  [L = requires OLLAMA_API_KEY]
   [L]  query_knowledge grounded in video content
   [L]  VideoSummarizationAgent  brief
   [L]  VideoSummarizationAgent  standard
   [L]  VideoSummarizationAgent  detailed
   [L]  Word-count table  (window / chapter / overall)
   [L]  YouTube URL ingestion + transcript search

  LLM key available : Tr